In [227]:
import pandas as pd
import numpy as np
from pathlib import Path

In [228]:
PROCESSED_DATA = Path("../data/processed")
RAW_DATA = Path("../data/raw")

cohort = pd.read_csv(PROCESSED_DATA / "cohort_9957.csv")

print("Cohort shape:", cohort.shape)
cohort.head()

Cohort shape: (9957, 158)


,SEQN,SDDSRVYR,RIDSTATR,RIDEXMON,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDAGEEX,RIDRETH1,RIDRETH2,...,FIAPROXY,FIAINTRP,MIALANG,MIAPROXY,MIAINTRP,AIALANG,ELIGSTAT,MORTSTAT,PERMTH_INT,death_5yr
0,2.0,1.0,2.0,2.0,1.0,77.0,926.0,926.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1,1,177,0
1,5.0,1.0,2.0,2.0,1.0,49.0,597.0,597.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1,0,244,0
2,7.0,1.0,2.0,2.0,2.0,59.0,712.0,712.0,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1,0,237,0
3,10.0,1.0,2.0,2.0,1.0,43.0,518.0,518.0,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1,1,231,0
4,13.0,1.0,2.0,2.0,1.0,70.0,850.0,850.0,1.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1,1,16,1


In [229]:
# Load Body Measures files for 1999-2004

bmx_1999 = pd.read_sas(RAW_DATA / "BMX.xpt", format="xport")
bmx_2001 = pd.read_sas(RAW_DATA / "BMX_B.xpt", format="xport")
bmx_2003 = pd.read_sas(RAW_DATA / "BMX_C.xpt", format="xport")

print("1999-2000:", bmx_1999.shape)
print("2001-2002:", bmx_2001.shape)
print("2003-2004:", bmx_2003.shape)

1999-2000: (9282, 38)
2001-2002: (10477, 26)
2003-2004: (9643, 33)


In [230]:
# Combine the three BMX survey cycles

bmx = pd.concat(
    [bmx_1999, bmx_2001, bmx_2003],
    ignore_index=True
)

print("Combined BMX shape:", bmx.shape)

Combined BMX shape: (29402, 45)


In [231]:
# Body-measure variables used in the Lin et al. study

bmx_cols = [
    "SEQN",
    "BMXBMI",     # Body mass index
    "BMXLEG",     # Upper leg length
    "BMXCALF",    # Calf circumference
    "BMXARML",    # Upper arm length
    "BMXARMC",    # Arm circumference
    "BMXWAIST",   # Waist circumference
    "BMXTHICR",   # Thigh circumference
    "BMXTRI",     # Triceps skinfold
    "BMXSUB",     # Subscapular skinfold
]

In [232]:
for col in bmx_cols:
    print(col, col in bmx.columns)

SEQN True
BMXBMI True
BMXLEG True
BMXCALF True
BMXARML True
BMXARMC True
BMXWAIST True
BMXTHICR True
BMXTRI True
BMXSUB True


In [233]:
# Keep only the BMX variables used in the Lin et al. study

bmx_features = bmx[bmx_cols].copy()

print("BMX rows:", len(bmx_features))
print("Unique SEQN:", bmx_features["SEQN"].nunique())
print("Duplicate SEQN:", bmx_features["SEQN"].duplicated().sum())

BMX rows: 29402
Unique SEQN: 29402
Duplicate SEQN: 0


In [234]:
# Merge BMX body-measure features into our final cohort

cohort = cohort.merge(
    bmx_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after BMX merge:", cohort.shape)

Cohort shape after BMX merge: (9957, 167)


In [235]:
print(cohort[
    [
        "SEQN",
        "BMXBMI",
        "BMXLEG",
        "BMXCALF",
        "BMXARML",
        "BMXARMC",
        "BMXWAIST",
        "BMXTHICR",
        "BMXTRI",
        "BMXSUB"
    ]
].head())

   SEQN  BMXBMI  BMXLEG  BMXCALF  BMXARML  BMXARMC  BMXWAIST  BMXTHICR  \
0   2.0   24.90     NaN     37.5     38.2     29.8      98.0       NaN   
1   5.0   29.10    45.2     42.6     39.7     35.8      99.9      56.2   
2   7.0   29.39    43.0     37.2     38.1     31.7      90.7      55.7   
3  10.0   30.94    46.6     43.7     43.0     37.6     108.0      64.0   
4  13.0   25.57    40.5     31.7     33.5     27.0       NaN       NaN   

   BMXTRI  BMXSUB  
0    12.8    20.4  
1    17.4    38.6  
2    26.4    34.2  
3    15.5    26.6  
4    15.0     8.8  


In [236]:
# Load Bioelectrical Impedance Analysis files for 1999-2004

bix_1999 = pd.read_sas(RAW_DATA / "BIX.xpt", format="xport")
bix_2001 = pd.read_sas(RAW_DATA / "BIX_B.xpt", format="xport")
bix_2003 = pd.read_sas(RAW_DATA / "BIX_C.xpt", format="xport")

print("1999-2000:", bix_1999.shape)
print("2001-2002:", bix_2001.shape)
print("2003-2004:", bix_2003.shape)

1999-2000: (5311, 115)
2001-2002: (5949, 115)
2003-2004: (5329, 115)


In [237]:
# Combine the three BIX survey cycles

bix = pd.concat(
    [bix_1999, bix_2001, bix_2003],
    ignore_index=True
)

print("Combined BIX shape:", bix.shape)

Combined BIX shape: (16589, 115)


In [238]:
bix_cols = [
    "SEQN",
    "BIDECF",   # Extracellular fluid
    "BIDTBW",   # Total body water
    "BIDICF",   # Intracellular fluid
    "BIDFFM",   # Fat-free mass
    "BIDFAT",   # Fat mass
]

In [239]:
for col in bix_cols:
    print(col, col in bix.columns)

SEQN True
BIDECF True
BIDTBW True
BIDICF True
BIDFFM True
BIDFAT True


In [240]:
# Keep only the BIX variables used in Lin et al.

bix_features = bix[bix_cols].copy()

print("BIX rows:", len(bix_features))
print("Unique SEQN:", bix_features["SEQN"].nunique())
print("Duplicate SEQN:", bix_features["SEQN"].duplicated().sum())

BIX rows: 16589
Unique SEQN: 16589
Duplicate SEQN: 0


In [241]:
# Merge BIX body-composition features into our cohort

cohort = cohort.merge(
    bix_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after BIX merge:", cohort.shape)

Cohort shape after BIX merge: (9957, 172)


In [242]:
print(
    cohort[
        [
            "SEQN",
            "BIDECF",
            "BIDTBW",
            "BIDICF",
            "BIDFFM",
            "BIDFAT",
        ]
    ].head()
)

   SEQN  BIDECF  BIDTBW  BIDICF  BIDFFM  BIDFAT
0   2.0     NaN     NaN     NaN     NaN     NaN
1   5.0   20.29   45.57   25.28    60.9    31.6
2   7.0     NaN     NaN     NaN     NaN     NaN
3  10.0   25.11   56.41   31.30    75.4    36.4
4  13.0     NaN     NaN     NaN     NaN     NaN


In [243]:
# Load Blood Pressure examination files for 1999-2004

bpx_1999 = pd.read_sas(RAW_DATA / "BPX.xpt", format="xport")
bpx_2001 = pd.read_sas(RAW_DATA / "BPX_B.xpt", format="xport")
bpx_2003 = pd.read_sas(RAW_DATA / "BPX_C.xpt", format="xport")

print("1999-2000:", bpx_1999.shape)
print("2001-2002:", bpx_2001.shape)
print("2003-2004:", bpx_2003.shape)

1999-2000: (9282, 30)
2001-2002: (10477, 30)
2003-2004: (9643, 30)


In [244]:
bpx = pd.concat(
    [bpx_1999, bpx_2001, bpx_2003],
    ignore_index=True
)

print("Combined BPX shape:", bpx.shape)

Combined BPX shape: (29402, 30)


In [245]:
bpx_cols = [
    "SEQN",
    "BPXPLS",   # Resting heart rate / pulse
]

In [246]:
for col in bpx_cols:
    print(col, col in bpx.columns)

SEQN True
BPXPLS True


In [247]:
# Keep only the BPX variable used in Lin et al.

bpx_features = bpx[bpx_cols].copy()

print("BPX rows:", len(bpx_features))
print("Unique SEQN:", bpx_features["SEQN"].nunique())
print("Duplicate SEQN:", bpx_features["SEQN"].duplicated().sum())

BPX rows: 29402
Unique SEQN: 29402
Duplicate SEQN: 0


In [248]:
# Merge resting heart rate into our cohort

cohort = cohort.merge(
    bpx_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after BPX merge:", cohort.shape)

Cohort shape after BPX merge: (9957, 173)


In [249]:
print(
    cohort[
        [
            "SEQN",
            "BPXPLS"
        ]
    ].head()
)

   SEQN  BPXPLS
0   2.0    68.0
1   5.0    66.0
2   7.0    58.0
3  10.0    58.0
4  13.0   102.0


In [250]:
# Load Cardiovascular Fitness files for 1999-2004

cvx_1999 = pd.read_sas(RAW_DATA / "CVX.xpt", format="xport")
cvx_2001 = pd.read_sas(RAW_DATA / "CVX_B.xpt", format="xport")
cvx_2003 = pd.read_sas(RAW_DATA / "CVX_C.xpt", format="xport")

print("1999-2000:", cvx_1999.shape)
print("2001-2002:", cvx_2001.shape)
print("2003-2004:", cvx_2003.shape)

1999-2000: (4602, 66)
2001-2002: (5152, 66)
2003-2004: (4663, 66)


In [251]:
cvx = pd.concat(
    [cvx_1999, cvx_2001, cvx_2003],
    ignore_index=True
)

print("Combined CVX shape:", cvx.shape)

Combined CVX shape: (14417, 66)


In [252]:
cvx_cols = [
    "SEQN",
    "CVDVOMAX",   # Predicted VO2max
    "CVDESVO2",   # Estimated VO2max
    "CVDFITLV",   # Cardiovascular fitness level
    "CVDR1HR",    # Heart rate recovery 1 min
    "CVDR2HR",    # Heart rate recovery 2 min
    "CVDR3HR",    # Heart rate recovery 3 min
]

In [253]:
for col in cvx_cols:
    print(col, col in cvx.columns)

SEQN True
CVDVOMAX True
CVDESVO2 True
CVDFITLV True
CVDR1HR True
CVDR2HR True
CVDR3HR True


In [254]:
# Keep only the CVX variables used in Lin et al.

cvx_features = cvx[cvx_cols].copy()

print("CVX rows:", len(cvx_features))
print("Unique SEQN:", cvx_features["SEQN"].nunique())
print("Duplicate SEQN:", cvx_features["SEQN"].duplicated().sum())

CVX rows: 14417
Unique SEQN: 14417
Duplicate SEQN: 0


In [255]:
# Merge cardiovascular fitness features into our cohort

cohort = cohort.merge(
    cvx_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after CVX merge:", cohort.shape)

Cohort shape after CVX merge: (9957, 179)


In [256]:
print(
    cohort[
        [
            "SEQN",
            "CVDVOMAX",
            "CVDESVO2",
            "CVDFITLV",
            "CVDR1HR",
            "CVDR2HR",
            "CVDR3HR",
        ]
    ].head()
)

   SEQN  CVDVOMAX  CVDESVO2  CVDFITLV  CVDR1HR  CVDR2HR  CVDR3HR
0   2.0       NaN       NaN       NaN      NaN      NaN      NaN
1   5.0     32.51     40.03       3.0    118.0    112.0    105.0
2   7.0       NaN       NaN       NaN      NaN      NaN      NaN
3  10.0       NaN       NaN       NaN      NaN      NaN      NaN
4  13.0       NaN       NaN       NaN      NaN      NaN      NaN


In [257]:
# Load Complete Blood Count files for 1999-2004

cbc_1999 = pd.read_sas(RAW_DATA / "LAB25.xpt", format="xport")
cbc_2001 = pd.read_sas(RAW_DATA / "L25_B.xpt", format="xport")
cbc_2003 = pd.read_sas(RAW_DATA / "L25_C.xpt", format="xport")

print("1999-2000:", cbc_1999.shape)
print("2001-2002:", cbc_2001.shape)
print("2003-2004:", cbc_2003.shape)

1999-2000: (8832, 21)
2001-2002: (9929, 21)
2003-2004: (9179, 21)


In [258]:
cbc = pd.concat(
    [cbc_1999, cbc_2001, cbc_2003],
    ignore_index=True
)

print("Combined CBC shape:", cbc.shape)

Combined CBC shape: (27940, 21)


In [259]:
cbc_cols = [
    "SEQN",
    "LBXWBCSI",  # White blood cell count
    "LBXLYPCT",  # Lymphocyte percentage
    "LBXMOPCT",  # Monocyte percentage
    "LBXNEPCT",  # Neutrophil percentage
    "LBXEOPCT",  # Eosinophil percentage
    "LBXBAPCT",  # Basophil percentage
    "LBDLYMNO",  # Lymphocyte count
    "LBDMONO",   # Monocyte count
    "LBDNENO",   # Neutrophil count
    "LBDEONO",   # Eosinophil count
    "LBDBANO",   # Basophil count
    "LBXRBCSI",  # Red blood cell count
    "LBXHGB",    # Hemoglobin
    "LBXHCT",    # Hematocrit
    "LBXMCVSI",  # Mean cell volume
    "LBXMCHSI",  # Mean cell hemoglobin
    "LBXMC",     # MCHC
    "LBXRDW",    # Red cell distribution width
    "LBXPLTSI",  # Platelet count
]

In [260]:
for col in cbc_cols:
    print(col, col in cbc.columns)

SEQN True
LBXWBCSI True
LBXLYPCT True
LBXMOPCT True
LBXNEPCT True
LBXEOPCT True
LBXBAPCT True
LBDLYMNO True
LBDMONO True
LBDNENO True
LBDEONO True
LBDBANO True
LBXRBCSI True
LBXHGB True
LBXHCT True
LBXMCVSI True
LBXMCHSI True
LBXMC True
LBXRDW True
LBXPLTSI True


In [261]:
# Keep only the CBC variables used in Lin et al.

cbc_features = cbc[cbc_cols].copy()

print("CBC rows:", len(cbc_features))
print("Unique SEQN:", cbc_features["SEQN"].nunique())
print("Duplicate SEQN:", cbc_features["SEQN"].duplicated().sum())

CBC rows: 27940
Unique SEQN: 27940
Duplicate SEQN: 0


In [262]:
# Merge CBC features into our cohort

cohort = cohort.merge(
    cbc_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after CBC merge:", cohort.shape)

Cohort shape after CBC merge: (9957, 198)


In [263]:
print(
    cohort[
        [
            "SEQN",
            "LBXWBCSI",
            "LBXLYPCT",
            "LBXHGB",
            "LBXRDW",
            "LBXPLTSI",
        ]
    ].head()
)

   SEQN  LBXWBCSI  LBXLYPCT  LBXHGB  LBXRDW  LBXPLTSI
0   2.0       7.6      21.1    14.1    13.7     214.0
1   5.0       5.9      37.8    14.5    13.1     209.0
2   7.0       4.9      46.8    13.4    14.3     244.0
3  10.0       4.6      41.3    15.4    13.7     167.0
4  13.0      11.6      13.1    16.8    14.4     228.0


In [264]:
# Load Standard Biochemistry files for 1999-2004

chem_1999 = pd.read_sas(RAW_DATA / "LAB18.xpt", format="xport")
chem_2001 = pd.read_sas(RAW_DATA / "L40_B.xpt", format="xport")
chem_2003 = pd.read_sas(RAW_DATA / "L40_C.xpt", format="xport")

print("1999-2000:", chem_1999.shape)
print("2001-2002:", chem_2001.shape)
print("2003-2004:", chem_2003.shape)

1999-2000: (6758, 41)
2001-2002: (7445, 41)
2003-2004: (6990, 37)


In [265]:
# Combine the three chemistry survey cycles

chem = pd.concat(
    [chem_1999, chem_2001, chem_2003],
    ignore_index=True
)

print("Combined chemistry shape:", chem.shape)

Combined chemistry shape: (21193, 46)


In [266]:
chem_cols = [
    "SEQN",
    "LBXSAL",     # Albumin
    "LBXSATSI",   # ALT
    "LBXSASSI",   # AST
    "LBXSAPSI",   # Alkaline phosphatase
    "LBXSBU",     # BUN
    "LBXSCA",     # Calcium
    "LBXSCH",     # Total cholesterol
    "LBXSGTSI",   # Gamma-GT
    "LBXSGL",     # Serum glucose
    "LBXSIR",     # Iron
    "LBXSLDSI",   # LDH
    "LBXSPH",     # Phosphorus
    "LBXSTB",     # Bilirubin
    "LBXSTP",     # Total protein
    "LBXSTR",     # Triglycerides
    "LBXSUA",     # Uric acid
    "LBXSNASI",   # Sodium
    "LBXSKSI",    # Potassium
    "LBXSCLSI",   # Chloride
    "LBXSOSSI",   # Osmolality
]

In [267]:
for col in chem_cols:
    print(col, col in chem.columns)

SEQN True
LBXSAL True
LBXSATSI True
LBXSASSI True
LBXSAPSI True
LBXSBU True
LBXSCA True
LBXSCH True
LBXSGTSI True
LBXSGL True
LBXSIR True
LBXSLDSI True
LBXSPH True
LBXSTB True
LBXSTP True
LBXSTR True
LBXSUA True
LBXSNASI True
LBXSKSI True
LBXSCLSI True
LBXSOSSI True


In [268]:
missing_chem_cols = [
    col for col in chem_cols
    if col not in chem.columns
]

print("Missing chemistry columns:", missing_chem_cols)

Missing chemistry columns: []


In [269]:
# Keep only the chemistry variables used in Lin et al.

chem_features = chem[chem_cols].copy()

print("Chemistry rows:", len(chem_features))
print("Unique SEQN:", chem_features["SEQN"].nunique())
print("Duplicate SEQN:", chem_features["SEQN"].duplicated().sum())

Chemistry rows: 21193
Unique SEQN: 21193
Duplicate SEQN: 0


In [270]:
# Merge chemistry features into our cohort

cohort = cohort.merge(
    chem_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after chemistry merge:", cohort.shape)

Cohort shape after chemistry merge: (9957, 218)


In [271]:
# Load Glycohemoglobin / HbA1c files for 1999-2004

a1c_1999 = pd.read_sas(RAW_DATA / "LAB10.xpt", format="xport")
a1c_2001 = pd.read_sas(RAW_DATA / "L10_B.xpt", format="xport")
a1c_2003 = pd.read_sas(RAW_DATA / "L10_C.xpt", format="xport")

print("1999-2000:", a1c_1999.shape)
print("2001-2002:", a1c_2001.shape)
print("2003-2004:", a1c_2003.shape)

1999-2000: (6758, 2)
2001-2002: (7445, 2)
2003-2004: (6990, 2)


In [272]:
a1c = pd.concat(
    [a1c_1999, a1c_2001, a1c_2003],
    ignore_index=True
)

print("Combined A1c shape:", a1c.shape)

Combined A1c shape: (21193, 2)


In [273]:
a1c_cols = [
    "SEQN",
    "LBXGH",   # Glycohemoglobin / HbA1c
]

In [274]:
for col in a1c_cols:
    print(col, col in a1c.columns)

SEQN True
LBXGH True


In [275]:
a1c_features = a1c[a1c_cols].copy()

print("A1c rows:", len(a1c_features))
print("Unique SEQN:", a1c_features["SEQN"].nunique())
print("Duplicate SEQN:", a1c_features["SEQN"].duplicated().sum())

A1c rows: 21193
Unique SEQN: 21193
Duplicate SEQN: 0


In [276]:
cohort = cohort.merge(
    a1c_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after A1c merge:", cohort.shape)

Cohort shape after A1c merge: (9957, 219)


In [277]:
# Load CRP / bone alkaline phosphatase files for 1999-2004

crp_1999 = pd.read_sas(RAW_DATA / "LAB11.xpt", format="xport")
crp_2001 = pd.read_sas(RAW_DATA / "L11_B.xpt", format="xport")
crp_2003 = pd.read_sas(RAW_DATA / "L11_C.xpt", format="xport")

print("1999-2000:", crp_1999.shape)
print("2001-2002:", crp_2001.shape)
print("2003-2004:", crp_2003.shape)

1999-2000: (8344, 7)
2001-2002: (9262, 6)
2003-2004: (9179, 4)


In [278]:
for name, df in [
    ("1999-2000", crp_1999),
    ("2001-2002", crp_2001),
    ("2003-2004", crp_2003),
]:
    print("\n", name)

    for col in df.columns:
        if "CRP" in col.upper() or "AP" in col.upper():
            print(col)


 1999-2000
LBXCRP
LBXBAP

 2001-2002
LBXCRP
LBDBAP

 2003-2004
LBXCRP
LBXBAP


In [279]:
for name, df in [
    ("1999-2000", crp_1999),
    ("2001-2002", crp_2001),
    ("2003-2004", crp_2003),
]:
    print("\n", name)

    for col in df.columns:
        if "CRP" in col.upper() or "AP" in col.upper():
            print(col)


 1999-2000
LBXCRP
LBXBAP

 2001-2002
LBXCRP
LBDBAP

 2003-2004
LBXCRP
LBXBAP


In [280]:
# Standardize the bone alkaline phosphatase column name

crp_2001 = crp_2001.rename(
    columns={"LBDBAP": "LBXBAP"}
)

In [281]:
crp = pd.concat(
    [crp_1999, crp_2001, crp_2003],
    ignore_index=True
)

print("Combined CRP/BAP shape:", crp.shape)

Combined CRP/BAP shape: (26785, 9)


In [282]:
crp_cols = [
    "SEQN",
    "LBXCRP",   # C-reactive protein
    "LBXBAP",   # Bone alkaline phosphatase
]

crp_features = crp[crp_cols].copy()

In [283]:
print("CRP/BAP rows:", len(crp_features))
print("Unique SEQN:", crp_features["SEQN"].nunique())
print("Duplicate SEQN:", crp_features["SEQN"].duplicated().sum())

CRP/BAP rows: 26785
Unique SEQN: 26785
Duplicate SEQN: 0


In [284]:
cohort = cohort.merge(
    crp_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after CRP/BAP merge:", cohort.shape)

Cohort shape after CRP/BAP merge: (9957, 221)


In [285]:
# Load fasting glucose / C-peptide / insulin files

fast_1999 = pd.read_sas(RAW_DATA / "LAB10AM.xpt", format="xport")
fast_2001 = pd.read_sas(RAW_DATA / "L10AM_B.xpt", format="xport")
fast_2003 = pd.read_sas(RAW_DATA / "L10AM_C.xpt", format="xport")

print("1999-2000:", fast_1999.shape)
print("2001-2002:", fast_2001.shape)
print("2003-2004:", fast_2003.shape)

1999-2000: (3267, 8)
2001-2002: (3666, 8)
2003-2004: (3356, 7)


In [286]:
fast = pd.concat(
    [fast_1999, fast_2001, fast_2003],
    ignore_index=True
)

print("Combined fasting shape:", fast.shape)

Combined fasting shape: (10289, 10)


In [287]:
fast_cols = [
    "SEQN",
    "LBXGLU",    # Plasma glucose
    "LBXCPSI",   # C-peptide
    "LBXIN",     # Insulin
]

In [288]:
for col in fast_cols:
    print(col, col in fast.columns)

SEQN True
LBXGLU True
LBXCPSI True
LBXIN True


In [289]:
fast_features = fast[fast_cols].copy()

print("Fasting rows:", len(fast_features))
print("Unique SEQN:", fast_features["SEQN"].nunique())
print("Duplicate SEQN:", fast_features["SEQN"].duplicated().sum())

Fasting rows: 10289
Unique SEQN: 10289
Duplicate SEQN: 0


In [290]:
cohort = cohort.merge(
    fast_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after fasting merge:", cohort.shape)

Cohort shape after fasting merge: (9957, 224)


In [291]:
# Load HDL cholesterol files

hdl_1999 = pd.read_sas(RAW_DATA / "LAB13.xpt", format="xport")
hdl_2001 = pd.read_sas(RAW_DATA / "L13_B.xpt", format="xport")
hdl_2003 = pd.read_sas(RAW_DATA / "L13_C.xpt", format="xport")

print("HDL 1999-2000:", hdl_1999.shape)
print("HDL 2001-2002:", hdl_2001.shape)
print("HDL 2003-2004:", hdl_2003.shape)

HDL 1999-2000: (8344, 5)
HDL 2001-2002: (9262, 5)
HDL 2003-2004: (8556, 5)


In [292]:
# Load fasting triglyceride / lipid files

lipid_1999 = pd.read_sas(RAW_DATA / "LAB13AM.xpt", format="xport")
lipid_2001 = pd.read_sas(RAW_DATA / "L13AM_B.xpt", format="xport")
lipid_2003 = pd.read_sas(RAW_DATA / "L13AM_C.xpt", format="xport")

print("Fasting lipids 1999-2000:", lipid_1999.shape)
print("Fasting lipids 2001-2002:", lipid_2001.shape)
print("Fasting lipids 2003-2004:", lipid_2003.shape)

Fasting lipids 1999-2000: (3915, 7)
Fasting lipids 2001-2002: (4464, 7)
Fasting lipids 2003-2004: (4034, 6)


In [293]:
# Check exact HDL variable names in each cycle

for name, df in [
    ("HDL 1999-2000", hdl_1999),
    ("HDL 2001-2002", hdl_2001),
    ("HDL 2003-2004", hdl_2003),
]:
    print("\n", name)
    print(df.columns.tolist())


 HDL 1999-2000
['SEQN', 'LBXTC', 'LBDTCSI', 'LBDHDL', 'LBDHDLSI']

 HDL 2001-2002
['SEQN', 'LBXTC', 'LBDHDL', 'LBDTCSI', 'LBDHDLSI']

 HDL 2003-2004
['SEQN', 'LBXTC', 'LBXHDD', 'LBDTCSI', 'LBDHDDSI']


In [294]:
# Check exact fasting lipid variable names in each cycle

for name, df in [
    ("Lipids 1999-2000", lipid_1999),
    ("Lipids 2001-2002", lipid_2001),
    ("Lipids 2003-2004", lipid_2003),
]:
    print("\n", name)
    print(df.columns.tolist())


 Lipids 1999-2000
['SEQN', 'WTSAF2YR', 'WTSAF4YR', 'LBXTR', 'LBDTRSI', 'LBDLDL', 'LBDLDLSI']

 Lipids 2001-2002
['SEQN', 'WTSAF2YR', 'WTSAF4YR', 'LBXTR', 'LBDTRSI', 'LBDLDL', 'LBDLDLSI']

 Lipids 2003-2004
['SEQN', 'WTSAF2YR', 'LBXTR', 'LBDTRSI', 'LBDLDL', 'LBDLDLSI']


In [295]:
# Standardize HDL column name across cycles

hdl_2003 = hdl_2003.rename(
    columns={"LBXHDD": "LBDHDL"}
)

In [296]:
hdl = pd.concat(
    [hdl_1999, hdl_2001, hdl_2003],
    ignore_index=True
)

hdl_cols = [
    "SEQN",
    "LBDHDL",   # HDL cholesterol
]

hdl_features = hdl[hdl_cols].copy()

print("HDL rows:", len(hdl_features))
print("Unique SEQN:", hdl_features["SEQN"].nunique())
print("Duplicate SEQN:", hdl_features["SEQN"].duplicated().sum())

HDL rows: 26162
Unique SEQN: 26162
Duplicate SEQN: 0


In [297]:
lipid = pd.concat(
    [lipid_1999, lipid_2001, lipid_2003],
    ignore_index=True
)

lipid_cols = [
    "SEQN",
    "LBXTR",    # Triglycerides
    "LBDLDL",   # LDL cholesterol
]

lipid_features = lipid[lipid_cols].copy()

print("Lipid rows:", len(lipid_features))
print("Unique SEQN:", lipid_features["SEQN"].nunique())
print("Duplicate SEQN:", lipid_features["SEQN"].duplicated().sum())

Lipid rows: 12413
Unique SEQN: 12413
Duplicate SEQN: 0


In [298]:
# Merge HDL cholesterol into our cohort

cohort = cohort.merge(
    hdl_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after HDL merge:", cohort.shape)

Cohort shape after HDL merge: (9957, 225)


In [299]:
# Merge triglycerides and LDL cholesterol into our cohort

cohort = cohort.merge(
    lipid_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after lipid merge:", cohort.shape)

Cohort shape after lipid merge: (9957, 227)


In [300]:
print(
    cohort[
        [
            "SEQN",
            "LBDHDL",   # HDL
            "LBXTR",    # Triglycerides
            "LBDLDL",   # LDL
        ]
    ].head()
)

   SEQN  LBDHDL  LBXTR  LBDLDL
0   2.0    54.0  128.0   136.0
1   5.0    42.0  347.0   168.0
2   7.0   105.0   62.0   127.0
3  10.0    51.0   45.0    80.0
4  13.0    49.0    NaN     NaN


In [301]:
# Load Troponin T and NT-proBNP files

troponin = pd.read_sas(
    RAW_DATA / "SSTROP_A.xpt",
    format="xport"
)

bnp = pd.read_sas(
    RAW_DATA / "SSBNP_A.xpt",
    format="xport"
)

print("Troponin file:", troponin.shape)
print("NT-proBNP file:", bnp.shape)

Troponin file: (21743, 12)
NT-proBNP file: (21206, 6)


In [302]:
print("SSTNT exists:", "SSTNT" in troponin.columns)
print("SSBNP exists:", "SSBNP" in bnp.columns)

SSTNT exists: True
SSBNP exists: True


In [303]:
# Keep only the cardiac biomarker variables we need

troponin_features = troponin[
    ["SEQN", "SSTNT"]
].copy()

bnp_features = bnp[
    ["SEQN", "SSBNP"]
].copy()

print("Troponin duplicate SEQN:",
      troponin_features["SEQN"].duplicated().sum())

print("BNP duplicate SEQN:",
      bnp_features["SEQN"].duplicated().sum())

Troponin duplicate SEQN: 0
BNP duplicate SEQN: 0


In [304]:
# Merge Troponin T

cohort = cohort.merge(
    troponin_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("After Troponin merge:", cohort.shape)

After Troponin merge: (9957, 228)


In [305]:
# Merge NT-proBNP

cohort = cohort.merge(
    bnp_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("After NT-proBNP merge:", cohort.shape)

After NT-proBNP merge: (9957, 229)


In [306]:
print(
    cohort[
        [
            "SEQN",
            "SSTNT",
            "SSBNP"
        ]
    ].head()
)

   SEQN  SSTNT   SSBNP
0   2.0   9.08   75.45
1   5.0   6.26   28.08
2   7.0   4.54   13.71
3  10.0   4.84   39.28
4  13.0  28.72  176.30


In [307]:
# Load kidney biomarker surplus-specimen file

kidney = pd.read_sas(
    RAW_DATA / "SSCARD_A.xpt",
    format="xport"
)

print("Kidney biomarker file:", kidney.shape)

Kidney biomarker file: (21890, 13)


In [308]:
print("SSB2M exists:", "SSB2M" in kidney.columns)
print("SSCYST exists:", "SSCYST" in kidney.columns)

SSB2M exists: True
SSCYST exists: True


In [309]:
kidney_features = kidney[
    [
        "SEQN",
        "SSB2M",    # Beta-2 microglobulin
        "SSCYST",   # Cystatin C
    ]
].copy()

print("Kidney rows:", len(kidney_features))
print("Unique SEQN:", kidney_features["SEQN"].nunique())
print("Duplicate SEQN:", kidney_features["SEQN"].duplicated().sum())

Kidney rows: 21890
Unique SEQN: 21890
Duplicate SEQN: 0


In [310]:
cohort = cohort.merge(
    kidney_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after kidney merge:", cohort.shape)

Cohort shape after kidney merge: (9957, 231)


In [311]:
# Load serum cotinine files

cot_1999 = pd.read_sas(RAW_DATA / "LAB06.xpt", format="xport")
cot_2001 = pd.read_sas(RAW_DATA / "L06_B.xpt", format="xport")
cot_2003 = pd.read_sas(RAW_DATA / "L06COT_C.xpt", format="xport")

print("1999-2000:", cot_1999.shape)
print("2001-2002:", cot_2001.shape)
print("2003-2004:", cot_2003.shape)

1999-2000: (8832, 41)
2001-2002: (9929, 22)
2003-2004: (8556, 2)


In [312]:
cot = pd.concat(
    [cot_1999, cot_2001, cot_2003],
    ignore_index=True
)

print("Combined cotinine shape:", cot.shape)

Combined cotinine shape: (27317, 43)


In [313]:
cot_cols = [
    "SEQN",
    "LBXCOT",   # Serum cotinine
]

for col in cot_cols:
    print(col, col in cot.columns)

SEQN True
LBXCOT True


In [314]:
cot_features = cot[cot_cols].copy()

print("Cotinine rows:", len(cot_features))
print("Unique SEQN:", cot_features["SEQN"].nunique())
print("Duplicate SEQN:", cot_features["SEQN"].duplicated().sum())

Cotinine rows: 27317
Unique SEQN: 27317
Duplicate SEQN: 0


In [315]:
cohort = cohort.merge(
    cot_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after cotinine merge:", cohort.shape)

Cohort shape after cotinine merge: (9957, 232)


In [316]:
# Load Blood Pressure / Cholesterol questionnaire files

bpq_1999 = pd.read_sas(RAW_DATA / "BPQ.xpt", format="xport")
bpq_2001 = pd.read_sas(RAW_DATA / "BPQ_B.xpt", format="xport")
bpq_2003 = pd.read_sas(RAW_DATA / "BPQ_C.xpt", format="xport")

print("1999-2000:", bpq_1999.shape)
print("2001-2002:", bpq_2001.shape)
print("2003-2004:", bpq_2003.shape)

1999-2000: (6044, 36)
2001-2002: (6634, 36)
2003-2004: (6213, 36)


In [317]:
bpq = pd.concat(
    [bpq_1999, bpq_2001, bpq_2003],
    ignore_index=True
)

print("Combined BPQ shape:", bpq.shape)

Combined BPQ shape: (18891, 42)


In [318]:
bpq_cols = [
    "SEQN",
    "BPQ020",   # Ever told had high blood pressure
    "BPQ050A",  # Now taking prescribed BP medicine
    "BPQ080",   # Ever told cholesterol was high
    "BPQ100D",  # Now taking prescribed cholesterol medicine
]

In [319]:
for col in bpq_cols:
    print(col, col in bpq.columns)

SEQN True
BPQ020 True
BPQ050A True
BPQ080 True
BPQ100D True


In [320]:
# Keep the BPQ categorical variables used in the study

bpq_features = bpq[bpq_cols].copy()

print("BPQ rows:", len(bpq_features))
print("Unique SEQN:", bpq_features["SEQN"].nunique())
print("Duplicate SEQN:", bpq_features["SEQN"].duplicated().sum())

BPQ rows: 18891
Unique SEQN: 18891
Duplicate SEQN: 0


In [321]:
# Merge hypertension / cholesterol questionnaire features

cohort = cohort.merge(
    bpq_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after BPQ merge:", cohort.shape)

Cohort shape after BPQ merge: (9957, 236)


In [322]:
# Load Medical Conditions questionnaire files

mcq_1999 = pd.read_sas(RAW_DATA / "MCQ.xpt", format="xport")
mcq_2001 = pd.read_sas(RAW_DATA / "MCQ_B.xpt", format="xport")
mcq_2003 = pd.read_sas(RAW_DATA / "MCQ_C.xpt", format="xport")

print("1999-2000:", mcq_1999.shape)
print("2001-2002:", mcq_2001.shape)
print("2003-2004:", mcq_2003.shape)

1999-2000: (9493, 167)
2001-2002: (10470, 148)
2003-2004: (9645, 154)


In [323]:
mcq = pd.concat(
    [mcq_1999, mcq_2001, mcq_2003],
    ignore_index=True
)

print("Combined MCQ shape:", mcq.shape)

Combined MCQ shape: (29608, 183)


In [324]:
mcq_cols = [
    "SEQN",
    "MCQ160B",   # Congestive heart failure
    "MCQ160C",   # Coronary heart disease
    "MCQ160E",   # Heart attack / myocardial infarction
    "MCQ160F",   # Stroke
]

In [325]:
for col in mcq_cols:
    print(col, col in mcq.columns)

SEQN True
MCQ160B True
MCQ160C True
MCQ160E True
MCQ160F True


In [326]:
mcq_features = mcq[mcq_cols].copy()

print("MCQ rows:", len(mcq_features))
print("Unique SEQN:", mcq_features["SEQN"].nunique())
print("Duplicate SEQN:", mcq_features["SEQN"].duplicated().sum())

MCQ rows: 29608
Unique SEQN: 29608
Duplicate SEQN: 0


In [327]:
cohort = cohort.merge(
    mcq_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after MCQ merge:", cohort.shape)

Cohort shape after MCQ merge: (9957, 240)


In [328]:
# Load Diabetes questionnaire files

diq_1999 = pd.read_sas(RAW_DATA / "DIQ.xpt", format="xport")
diq_2001 = pd.read_sas(RAW_DATA / "DIQ_B.xpt", format="xport")
diq_2003 = pd.read_sas(RAW_DATA / "DIQ_C.xpt", format="xport")

print("1999-2000:", diq_1999.shape)
print("2001-2002:", diq_2001.shape)
print("2003-2004:", diq_2003.shape)

1999-2000: (9493, 17)
2001-2002: (10470, 17)
2003-2004: (9645, 17)


In [329]:
diq = pd.concat(
    [diq_1999, diq_2001, diq_2003],
    ignore_index=True
)

print("Combined DIQ shape:", diq.shape)

Combined DIQ shape: (29608, 21)


In [330]:
diq_cols = [
    "SEQN",
    "DIQ010",   # Diabetes history
    "DIQ070",   # Taking diabetes pills
]

In [331]:
for col in diq_cols:
    print(col, col in diq.columns)

SEQN True
DIQ010 True
DIQ070 True


In [332]:
diq_features = diq[diq_cols].copy()

print("DIQ rows:", len(diq_features))
print("Unique SEQN:", diq_features["SEQN"].nunique())
print("Duplicate SEQN:", diq_features["SEQN"].duplicated().sum())

DIQ rows: 29608
Unique SEQN: 29608
Duplicate SEQN: 0


In [333]:
cohort = cohort.merge(
    diq_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after DIQ merge:", cohort.shape)

Cohort shape after DIQ merge: (9957, 242)


In [334]:
# Load Physical Activity questionnaire files

paq_1999 = pd.read_sas(RAW_DATA / "PAQ.xpt", format="xport")
paq_2001 = pd.read_sas(RAW_DATA / "PAQ_B.xpt", format="xport")
paq_2003 = pd.read_sas(RAW_DATA / "PAQ_C.xpt", format="xport")

print("1999-2000:", paq_1999.shape)
print("2001-2002:", paq_2001.shape)
print("2003-2004:", paq_2003.shape)

1999-2000: (9188, 21)
2001-2002: (10094, 21)
2003-2004: (9278, 20)


In [335]:
paq = pd.concat(
    [paq_1999, paq_2001, paq_2003],
    ignore_index=True
)

print("Combined PAQ shape:", paq.shape)

Combined PAQ shape: (28560, 24)


In [336]:
paq_cols = [
    "SEQN",
    "PAQ180",   # Usual daily physical activity level
]

In [337]:
for col in paq_cols:
    print(col, col in paq.columns)

SEQN True
PAQ180 True


In [338]:
paq_features = paq[paq_cols].copy()

print("PAQ rows:", len(paq_features))
print("Unique SEQN:", paq_features["SEQN"].nunique())
print("Duplicate SEQN:", paq_features["SEQN"].duplicated().sum())

PAQ rows: 28560
Unique SEQN: 28560
Duplicate SEQN: 0


In [339]:
cohort = cohort.merge(
    paq_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after PAQ merge:", cohort.shape)

Cohort shape after PAQ merge: (9957, 243)


In [340]:
# Check remaining heart-rate variables from the CVX file
# using only our final 9,957-person cohort

cvx_hr = cvx[
    [
        "SEQN",
        "CVDWHR",    # Warm-up heart rate
        "CVDS1HR",   # Stage 1 heart rate
        "CVDS2HR",   # Stage 2 heart rate
    ]
].copy()

cvx_hr = cohort[["SEQN"]].merge(
    cvx_hr,
    on="SEQN",
    how="left"
)

print("Warm-up HR valid:", cvx_hr["CVDWHR"].notna().sum())
print("Stage 1 HR valid:", cvx_hr["CVDS1HR"].notna().sum())
print("Stage 2 HR valid:", cvx_hr["CVDS2HR"].notna().sum())

print(
    "Stage 1 + Stage 2 both valid:",
    cvx_hr[["CVDS1HR", "CVDS2HR"]].notna().all(axis=1).sum()
)

Warm-up HR valid: 922
Stage 1 HR valid: 1022
Stage 2 HR valid: 933
Stage 1 + Stage 2 both valid: 932


In [341]:
cvx_pmhr = cohort[["SEQN"]].merge(
    cvx[["SEQN", "CVDPMHR"]],
    on="SEQN",
    how="left"
)

print(
    "Predicted maximum heart rate valid:",
    cvx_pmhr["CVDPMHR"].notna().sum()
)

Predicted maximum heart rate valid: 2365


In [342]:
# Look at every CVX heart-rate variable
# and count how many values exist in our final cohort

hr_cols = [
    col for col in cvx.columns
    if "HR" in col.upper()
]

cvx_hr_check = cohort[["SEQN"]].merge(
    cvx[["SEQN"] + hr_cols],
    on="SEQN",
    how="left"
)

for col in hr_cols:
    print(
        col,
        cvx_hr_check[col].notna().sum()
    )

CVDPMHR 2365
CVDWHR 922
CVDS1HR 1022
CVDS2HR 933
CVDR1HR 950
CVDR2HR 995
CVDR3HR 471


In [343]:
# Maximum measured heart rate during the exercise stages

cvx_hr["MAX_HR"] = cvx_hr[
    ["CVDS1HR", "CVDS2HR"]
].max(axis=1)

print(
    "Maximum heart rate valid:",
    cvx_hr["MAX_HR"].notna().sum()
)

cvx_hr[
    ["SEQN", "CVDS1HR", "CVDS2HR", "MAX_HR"]
].head()

Maximum heart rate valid: 1023


,SEQN,CVDS1HR,CVDS2HR,MAX_HR
0,2.0,NaN,NaN,NaN
1,5.0,107.0,126.0,126.0
2,7.0,NaN,NaN,NaN
3,10.0,NaN,NaN,NaN
4,13.0,NaN,NaN,NaN


In [344]:
# Keep the confirmed heart-rate features

hr_features = cvx_hr[
    [
        "SEQN",
        "CVDWHR",   # Heart rate before exercise
        "MAX_HR",   # Maximum measured exercise heart rate
    ]
].copy()

print("Duplicate SEQN:",
      hr_features["SEQN"].duplicated().sum())

Duplicate SEQN: 0


In [345]:
cohort = cohort.merge(
    hr_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after heart-rate merge:", cohort.shape)

Cohort shape after heart-rate merge: (9957, 245)


In [346]:
# Load telomere length files

telo_1999 = pd.read_sas(
    RAW_DATA / "TELO_A.xpt",
    format="xport"
)

telo_2001 = pd.read_sas(
    RAW_DATA / "TELO_B.xpt",
    format="xport"
)

print("1999-2000:", telo_1999.shape)
print("2001-2002:", telo_2001.shape)

1999-2000: (3570, 3)
2001-2002: (4269, 3)


In [347]:
print("1999-2000 columns:")
print(telo_1999.columns.tolist())

print("\n2001-2002 columns:")
print(telo_2001.columns.tolist())

1999-2000 columns:
['SEQN', 'TELOMEAN', 'TELOSTD']

2001-2002 columns:
['SEQN', 'TELOMEAN', 'TELOSTD']


In [348]:
# Combine telomere files

telo = pd.concat(
    [telo_1999, telo_2001],
    ignore_index=True
)

print("Combined telomere shape:", telo.shape)


Combined telomere shape: (7839, 3)


In [349]:
telo_cols = [
    "SEQN",
    "TELOMEAN",
    "TELOSTD",
]

telo_features = telo[telo_cols].copy()

In [350]:
print("Telomere rows:", len(telo_features))
print("Unique SEQN:", telo_features["SEQN"].nunique())
print("Duplicate SEQN:", telo_features["SEQN"].duplicated().sum())

Telomere rows: 7839
Unique SEQN: 7839
Duplicate SEQN: 0


In [351]:
cohort = cohort.merge(
    telo_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after telomere merge:", cohort.shape)

Cohort shape after telomere merge: (9957, 247)


In [352]:
# Load sex hormone files

horm_1999 = pd.read_sas(
    RAW_DATA / "SSCHL_A.xpt",
    format="xport"
)

horm_2001 = pd.read_sas(
    RAW_DATA / "SSCHL_B.xpt",
    format="xport"
)

horm_2003 = pd.read_sas(
    RAW_DATA / "SSCHL_C.xpt",
    format="xport"
)

print("1999-2000:", horm_1999.shape)
print("2001-2002:", horm_2001.shape)
print("2003-2004:", horm_2003.shape)

1999-2000: (414, 5)
2001-2002: (526, 5)
2003-2004: (490, 5)


In [353]:
print("1999-2000 columns:")
print(horm_1999.columns.tolist())

print("\n2001-2002 columns:")
print(horm_2001.columns.tolist())

print("\n2003-2004 columns:")
print(horm_2003.columns.tolist())

1999-2000 columns:
['SEQN', 'SSTESTO', 'SSSHBG', 'SSSE2', 'SS3ADLG']

2001-2002 columns:
['SEQN', 'SSTESTO', 'SSSHBG', 'SSSE2', 'SS3ADLG']

2003-2004 columns:
['SEQN', 'SSTESTO', 'SSSHBG', 'SSSE2', 'SS3ADLG']


In [354]:
# Combine hormone files

horm = pd.concat(
    [horm_1999, horm_2001, horm_2003],
    ignore_index=True
)

print("Combined hormone shape:", horm.shape)

Combined hormone shape: (1430, 5)


In [355]:
horm_cols = [
    "SEQN",
    "SSTESTO",
    "SSSHBG",
    "SSSE2",
    "SS3ADLG",
]

horm_features = horm[horm_cols].copy()

In [356]:
print("Hormone rows:", len(horm_features))
print("Unique SEQN:", horm_features["SEQN"].nunique())
print("Duplicate SEQN:", horm_features["SEQN"].duplicated().sum())

Hormone rows: 1430
Unique SEQN: 1430
Duplicate SEQN: 0


In [357]:
# Merge hormone features into cohort

cohort = cohort.merge(
    horm_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after hormone merge:", cohort.shape)

Cohort shape after hormone merge: (9957, 251)


In [358]:
# Load blood metals files

metals_1999 = pd.read_sas(
    RAW_DATA / "LAB06.xpt",
    format="xport"
)

metals_2001 = pd.read_sas(
    RAW_DATA / "L06_B.xpt",
    format="xport"
)

metals_2003 = pd.read_sas(
    RAW_DATA / "L06BMT_C.xpt",
    format="xport"
)

print("1999-2000:", metals_1999.shape)
print("2001-2002:", metals_2001.shape)
print("2003-2004:", metals_2003.shape)

1999-2000: (8832, 41)
2001-2002: (9929, 22)
2003-2004: (9179, 9)


In [359]:
metals = pd.concat(
    [metals_1999, metals_2001, metals_2003],
    ignore_index=True
)

print("Combined metals shape:", metals.shape)

Combined metals shape: (27940, 43)


In [360]:
metals_cols = [
    "SEQN",
    "LBXBCD",   # Blood cadmium
    "LBXBPB",   # Blood lead
    "LBXTHG",   # Total blood mercury
]

for col in metals_cols:
    print(col, col in metals.columns)

SEQN True
LBXBCD True
LBXBPB True
LBXTHG True


In [361]:
metals_features = metals[metals_cols].copy()

print("Metals rows:", len(metals_features))
print("Unique SEQN:", metals_features["SEQN"].nunique())
print("Duplicate SEQN:", metals_features["SEQN"].duplicated().sum())

Metals rows: 27940
Unique SEQN: 27940
Duplicate SEQN: 0


In [362]:
cohort = cohort.merge(
    metals_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after metals merge:", cohort.shape)

Cohort shape after metals merge: (9957, 254)


In [363]:
# Load Vitamin B12 / RBC folate files

vit_1999 = pd.read_sas(RAW_DATA / "LAB06.xpt", format="xport")
vit_2001 = pd.read_sas(RAW_DATA / "L06_B.xpt", format="xport")
vit_2003 = pd.read_sas(RAW_DATA / "L06NB_C.xpt", format="xport")

print("1999-2000:", vit_1999.shape)
print("2001-2002:", vit_2001.shape)
print("2003-2004:", vit_2003.shape)

1999-2000: (8832, 41)
2001-2002: (9929, 22)
2003-2004: (9179, 7)


In [364]:
vit = pd.concat(
    [vit_1999, vit_2001, vit_2003],
    ignore_index=True
)

print("Combined vitamin shape:", vit.shape)

Combined vitamin shape: (27940, 43)


In [365]:
vit_cols = [
    "SEQN",
    "LBXB12",   # Vitamin B12
    "LBXRBF",   # RBC folate
]

for col in vit_cols:
    print(col, col in vit.columns)

SEQN True
LBXB12 True
LBXRBF True


In [366]:
vit_features = vit[vit_cols].copy()

print("Vitamin rows:", len(vit_features))
print("Unique SEQN:", vit_features["SEQN"].nunique())
print("Duplicate SEQN:", vit_features["SEQN"].duplicated().sum())

Vitamin rows: 27940
Unique SEQN: 27940
Duplicate SEQN: 0


In [367]:
cohort = cohort.merge(
    vit_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after vitamin merge:", cohort.shape)

Cohort shape after vitamin merge: (9957, 256)


In [368]:
# Check glycated albumin variables already inside SSCARD_A

ga_cols = [
    "SSGALB",    # Glycated albumin
    "SSGALBP",   # Percent glycated albumin
    "SSALB",     # Albumin measured with this assay
]

for col in ga_cols:
    print(
        col,
        col in kidney.columns,
        kidney[col].notna().sum() if col in kidney.columns else None
    )

SSGALB True 21842
SSGALBP True 21815
SSALB True 21837


In [369]:
# Keep glycated albumin percentage

ga_features = kidney[
    [
        "SEQN",
        "SSGALBP",   # Percent glycated albumin
    ]
].copy()

print("Glycated albumin rows:", len(ga_features))
print("Unique SEQN:", ga_features["SEQN"].nunique())
print("Duplicate SEQN:", ga_features["SEQN"].duplicated().sum())

Glycated albumin rows: 21890
Unique SEQN: 21890
Duplicate SEQN: 0


In [370]:
cohort = cohort.merge(
    ga_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after glycated albumin merge:", cohort.shape)

Cohort shape after glycated albumin merge: (9957, 257)


In [371]:
# Load thyroid files

thy_1999 = pd.read_sas(
    RAW_DATA / "LAB18T4.xpt",
    format="xport"
)

thy_2001 = pd.read_sas(
    RAW_DATA / "L40T4_B.xpt",
    format="xport"
)

print("1999-2000:", thy_1999.shape)
print("2001-2002:", thy_2001.shape)

1999-2000: (2276, 6)
2001-2002: (2547, 6)


In [372]:
print("1999-2000 columns:")
print(thy_1999.columns.tolist())

print("\n2001-2002 columns:")
print(thy_2001.columns.tolist())

1999-2000 columns:
['SEQN', 'WTSTH2YR', 'WTSTH4YR', 'LBXT4', 'LBDT4SI', 'LBXTSH']

2001-2002 columns:
['SEQN', 'WTSTH2YR', 'WTSTH4YR', 'LBXT4', 'LBDT4SI', 'LBXTSH']


In [373]:
# Combine thyroid files

thy = pd.concat(
    [thy_1999, thy_2001],
    ignore_index=True
)

print("Combined thyroid shape:", thy.shape)

Combined thyroid shape: (4823, 6)


In [374]:
thy_cols = [
    "SEQN",
    "LBXT4",    # Thyroxine
    "LBXTSH",   # TSH
]

thy_features = thy[thy_cols].copy()

In [375]:
print("Thyroid rows:", len(thy_features))
print("Unique SEQN:", thy_features["SEQN"].nunique())
print("Duplicate SEQN:", thy_features["SEQN"].duplicated().sum())

Thyroid rows: 4823
Unique SEQN: 4823
Duplicate SEQN: 0


In [376]:
cohort = cohort.merge(
    thy_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after thyroid merge:", cohort.shape)

Cohort shape after thyroid merge: (9957, 259)


In [377]:
# Check globulin in the chemistry data

print("LBXSGB exists:", "LBXSGB" in chem.columns)

LBXSGB exists: True


In [378]:
# Keep globulin

globulin_features = chem[
    [
        "SEQN",
        "LBXSGB",   # Globulin (g/dL)
    ]
].copy()

print("Globulin rows:", len(globulin_features))
print("Unique SEQN:", globulin_features["SEQN"].nunique())
print("Duplicate SEQN:", globulin_features["SEQN"].duplicated().sum())

Globulin rows: 21193
Unique SEQN: 21193
Duplicate SEQN: 0


In [379]:
cohort = cohort.merge(
    globulin_features,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Cohort shape after globulin merge:", cohort.shape)

Cohort shape after globulin merge: (9957, 260)


In [380]:
# Check serum creatinine needed for eGFR

print("LBXSCR exists:", "LBXSCR" in chem.columns)

creatinine_check = cohort[
    ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH1"]
].merge(
    chem[["SEQN", "LBXSCR"]],
    on="SEQN",
    how="left"
)

print(
    "Creatinine valid:",
    creatinine_check["LBXSCR"].notna().sum()
)

creatinine_check.head()

LBXSCR exists: True
Creatinine valid: 5556


,SEQN,RIDAGEYR,RIAGENDR,RIDRETH1,LBXSCR
0,2.0,77.0,1.0,3.0,0.7
1,5.0,49.0,1.0,3.0,0.8
2,7.0,59.0,2.0,4.0,0.6
3,10.0,43.0,1.0,4.0,0.9
4,13.0,70.0,1.0,1.0,0.7


In [381]:
# Prepare serum creatinine for eGFR

# 1999-2000 uses LBXSCR
creat_1999 = chem_1999[
    ["SEQN", "LBXSCR"]
].copy()

# 2001-2002 uses LBDSCR
creat_2001 = chem_2001[
    ["SEQN", "LBDSCR"]
].copy()

# 2003-2004 uses LBXSCR
creat_2003 = chem_2003[
    ["SEQN", "LBXSCR"]
].copy()


# Standardize the column name across cycles
creat_2001 = creat_2001.rename(
    columns={"LBDSCR": "LBXSCR"}
)


# CDC correction for 1999-2000 creatinine
creat_1999["LBXSCR_STD"] = (
    1.013 * creat_1999["LBXSCR"] + 0.147
)

# No correction needed for 2001-2002 or 2003-2004
creat_2001["LBXSCR_STD"] = creat_2001["LBXSCR"]
creat_2003["LBXSCR_STD"] = creat_2003["LBXSCR"]


# Combine all cycles
creatinine = pd.concat(
    [
        creat_1999[["SEQN", "LBXSCR_STD"]],
        creat_2001[["SEQN", "LBXSCR_STD"]],
        creat_2003[["SEQN", "LBXSCR_STD"]],
    ],
    ignore_index=True
)

print("Creatinine rows:", len(creatinine))
print("Unique SEQN:", creatinine["SEQN"].nunique())
print("Duplicate SEQN:", creatinine["SEQN"].duplicated().sum())

Creatinine rows: 21193
Unique SEQN: 21193
Duplicate SEQN: 0


In [382]:
# Prepare standardized serum creatinine for merging
creatinine_merge = creatinine[
    ["SEQN", "LBXSCR_STD"]
].copy()

creatinine_merge = creatinine_merge.rename(
    columns={"LBXSCR_STD": "serum_creatinine"}
)

# Remove old version if this cell was previously run
if "serum_creatinine" in cohort.columns:
    cohort = cohort.drop(columns=["serum_creatinine"])

# Merge into cohort
cohort = cohort.merge(
    creatinine_merge,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))
print("Creatinine available:", cohort["serum_creatinine"].notna().sum())
print("Creatinine missing:", cohort["serum_creatinine"].isna().sum())

Cohort rows: 9957
Creatinine available: 8541
Creatinine missing: 1416


In [383]:
import numpy as np

def calculate_egfr(row):
    scr = row["serum_creatinine"]
    age = row["RIDAGEYR"]
    sex = row["RIAGENDR"]

    # Cannot calculate without required values
    if pd.isna(scr) or pd.isna(age) or pd.isna(sex):
        return np.nan

    # CKD-EPI 2021 parameters
    if sex == 2:  # Female
        k = 0.7
        alpha = -0.241
        sex_factor = 1.012
    else:         # Male
        k = 0.9
        alpha = -0.302
        sex_factor = 1.0

    egfr = (
        142
        * min(scr / k, 1) ** alpha
        * max(scr / k, 1) ** -1.200
        * 0.9938 ** age
        * sex_factor
    )

    return egfr


cohort["eGFR"] = cohort.apply(calculate_egfr, axis=1)

print("eGFR available:", cohort["eGFR"].notna().sum())
print("eGFR missing:", cohort["eGFR"].isna().sum())

cohort[
    ["SEQN", "RIDAGEYR", "RIAGENDR", "serum_creatinine", "eGFR"]
].head(10)

eGFR available: 8541
eGFR missing: 1416


/var/folders/_c/4261b1s51b7c1nqjypkgwysr0000gn/T/ipykernel_36977/526840521.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cohort["eGFR"] = cohort.apply(calculate_egfr, axis=1)


,SEQN,RIDAGEYR,RIAGENDR,serum_creatinine,eGFR
0,2.0,77.0,1.0,0.8561,89.303730
1,5.0,49.0,1.0,0.9574,97.211552
2,7.0,59.0,2.0,0.7548,90.955463
3,10.0,43.0,1.0,1.0587,89.435211
4,13.0,70.0,1.0,0.8561,93.277447
5,14.0,81.0,1.0,1.1600,63.277331
6,16.0,85.0,2.0,1.0587,51.555305
7,24.0,53.0,2.0,0.7548,94.413660
8,25.0,42.0,2.0,0.6535,112.517734
9,29.0,62.0,1.0,1.0587,79.467471


In [384]:
cohort = cohort.copy()

In [385]:
print(cohort["eGFR"].describe())

print("\nMissing eGFR:", cohort["eGFR"].isna().sum())
print("eGFR < 60:", (cohort["eGFR"] < 60).sum())
print("eGFR >= 60:", (cohort["eGFR"] >= 60).sum())

count    8541.000000
mean       83.949730
std        20.684467
min         2.653709
25%        71.687458
50%        86.726289
75%        99.549146
max       130.598540
Name: eGFR, dtype: float64

Missing eGFR: 1416
eGFR < 60: 1139
eGFR >= 60: 7402


In [386]:
# Create low-eGFR indicator while preserving missing values
cohort["low_eGFR"] = pd.Series(
    pd.NA,
    index=cohort.index,
    dtype="Int64"
)

mask = cohort["eGFR"].notna()

cohort.loc[mask, "low_eGFR"] = (
    cohort.loc[mask, "eGFR"] < 60
).astype("Int64")

print(cohort["low_eGFR"].value_counts(dropna=False))

low_eGFR
0       7402
<NA>    1416
1       1139
Name: count, dtype: Int64


In [387]:
cohort[
    ["SEQN", "RIDAGEYR", "RIAGENDR", "serum_creatinine", "eGFR"]
].sort_values("eGFR").head(10)

,SEQN,RIDAGEYR,RIAGENDR,serum_creatinine,eGFR
7115,22413.0,68.0,2.0,13.7000,2.653709
1281,3894.0,61.0,2.0,12.1004,3.217106
2930,9172.0,75.0,2.0,10.1757,3.630227
5182,16283.0,72.0,1.0,12.9000,3.717056
6429,20234.0,40.0,2.0,10.8000,4.201822
777,2275.0,79.0,2.0,8.6562,4.299463
4835,15185.0,58.0,2.0,9.1000,4.614009
3381,10597.0,75.0,1.0,10.1000,4.893484
622,1855.0,69.0,1.0,10.3783,4.916526
1042,3141.0,63.0,1.0,10.2770,5.163881


In [388]:
print("Creatinine <= 0:",
      (cohort["serum_creatinine"] <= 0).sum())

print("eGFR <= 0:",
      (cohort["eGFR"] <= 0).sum())

print("Missing consistency:",
      ((cohort["serum_creatinine"].isna()) ==
       (cohort["eGFR"].isna())).all())

Creatinine <= 0: 0
eGFR <= 0: 0
Missing consistency: True


In [389]:
bix_cols = [
    "SEQN",
    "BIDECF",
    "BIDTBW",
    "BIDICF",
    "BIDFFM",
    "BIDFAT"
]

bix_features = bix[bix_cols].copy()

print("BIX rows:", len(bix_features))
print("Unique SEQN:", bix_features["SEQN"].nunique())
print("Duplicate SEQN:", bix_features["SEQN"].duplicated().sum())

BIX rows: 16589
Unique SEQN: 16589
Duplicate SEQN: 0


In [390]:
# BIX variables
bix_vars = [
    "BIDECF",
    "BIDTBW",
    "BIDICF",
    "BIDFFM",
    "BIDFAT"
]

# Remove any old/duplicate BIX columns from previous merge attempts
cols_to_drop = []

for col in bix_vars:
    for version in [col, f"{col}_x", f"{col}_y"]:
        if version in cohort.columns:
            cols_to_drop.append(version)

cohort = cohort.drop(columns=cols_to_drop)

# Merge clean BIX features
cohort = cohort.merge(
    bix_features,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))

for col in bix_vars:
    print(
        col,
        "available:",
        cohort[col].notna().sum(),
        "| missing:",
        cohort[col].isna().sum()
    )

Cohort rows: 9957
BIDECF available: 1781 | missing: 8176
BIDTBW available: 1781 | missing: 8176
BIDICF available: 1781 | missing: 8176
BIDFFM available: 1781 | missing: 8176
BIDFAT available: 1781 | missing: 8176


In [391]:
# Check whether all BIX variables are missing/present together
cohort["has_BIX"] = cohort[bix_vars].notna().all(axis=1)

print(cohort["has_BIX"].value_counts(dropna=False))

# Check age distribution
print("\nAge with BIX:")
print(cohort.loc[cohort["has_BIX"], "RIDAGEYR"].describe())

print("\nAge without BIX:")
print(cohort.loc[~cohort["has_BIX"], "RIDAGEYR"].describe())

has_BIX
False    8176
True     1781
Name: count, dtype: int64

Age with BIX:
count    1781.000000
mean       44.319483
std         2.820791
min        40.000000
25%        42.000000
50%        44.000000
75%        47.000000
max        49.000000
Name: RIDAGEYR, dtype: float64

Age without BIX:
count    8176.000000
mean       65.736913
std        12.206326
min        40.000000
25%        56.000000
50%        66.000000
75%        76.000000
max        85.000000
Name: RIDAGEYR, dtype: float64


In [392]:
# See whether BIX availability differs by NHANES cycle
cycle_candidates = [
    c for c in cohort.columns
    if "cycle" in c.lower() or c == "SDDSRVYR"
]

print("Possible cycle columns:", cycle_candidates)

Possible cycle columns: ['SDDSRVYR']


In [393]:
# BIX eligibility in our age-40+ cohort
age_40_49 = cohort["RIDAGEYR"].between(40, 49)

print("Participants age 40-49:", age_40_49.sum())

print(
    "BIX available among age 40-49:",
    cohort.loc[age_40_49, "has_BIX"].sum()
)

print(
    "BIX missing among age 40-49:",
    (~cohort.loc[age_40_49, "has_BIX"]).sum()
)

print(
    "BIX available age 50+:",
    cohort.loc[cohort["RIDAGEYR"] >= 50, "has_BIX"].sum()
)

Participants age 40-49: 2473
BIX available among age 40-49: 1781
BIX missing among age 40-49: 692
BIX available age 50+: 0


In [394]:
pd.crosstab(
    cohort["SDDSRVYR"],
    cohort["has_BIX"],
    margins=True
)

has_BIX,False,True,All
SDDSRVYR,,,
1.0,2640,542,3182
2.0,2813,670,3483
3.0,2723,569,3292
All,8176,1781,9957


In [395]:
print([
    name for name in globals()
    if "cvx" in name.lower()
])

['cvx_1999', 'cvx_2001', 'cvx_2003', 'cvx', 'cvx_cols', 'cvx_features', 'cvx_hr', 'cvx_pmhr', 'cvx_hr_check', 'cvx_target', 'cvx_vars', 'cvx_any']


In [396]:
print(cvx_1999.columns.tolist())
print(cvx_2001.columns.tolist())
print(cvx_2003.columns.tolist())

['SEQN', 'CVDEXSTS', 'CVDEXCMT', 'CVDEXCL1', 'CVDEXCL2', 'CVDEXCL3', 'CVDEXCL4', 'CVDEXCL5', 'CVDEXCL6', 'CVQ220A', 'CVQ220B', 'CVQ220C', 'CVQ220E', 'CVQ220G', 'CVQ220H', 'CVQ220I', 'CVQ220J', 'CVQ220K', 'CVQ220L', 'CVQ220M', 'CVDEXLEN', 'CVDEXMER', 'CVDPMHR', 'CVXPARC', 'CVDVOMAX', 'CVAPROT', 'CVDPROT', 'CVATEMP', 'CVAARM', 'CVACUFF', 'CVAVEST', 'CVAWS1', 'CVAWG1', 'CVDWTIM', 'CVXWRPE', 'CVDWHR', 'CVDWSY', 'CVDWDI', 'CVDS1S', 'CVDS1G', 'CVDS1TIM', 'CVXS1RPE', 'CVDS1HR', 'CVDS1SY', 'CVDS1DI', 'CVDS2S', 'CVDS2G', 'CVDS2TIM', 'CVXS2RPE', 'CVDS2HR', 'CVDS2SY', 'CVDS2DI', 'CVDR1TIM', 'CVDR1HR', 'CVDR1SY', 'CVDR1DI', 'CVDR2TIM', 'CVDR2HR', 'CVDR2SY', 'CVDR2DI', 'CVDR3TIM', 'CVDR3HR', 'CVDR3SY', 'CVDR3DI', 'CVDESVO2', 'CVDFITLV']
['SEQN', 'CVDEXSTS', 'CVDEXCMT', 'CVDEXCL1', 'CVDEXCL2', 'CVDEXCL3', 'CVDEXCL4', 'CVDEXCL5', 'CVDEXCL6', 'CVQ220A', 'CVQ220B', 'CVQ220C', 'CVQ220E', 'CVQ220G', 'CVQ220H', 'CVQ220I', 'CVQ220J', 'CVQ220K', 'CVQ220L', 'CVQ220M', 'CVDEXLEN', 'CVDEXMER', 'CVDPMHR', 'CVXP

In [397]:
cvx_target = [
    "SEQN",
    "CVDVOMAX",
    "CVDWHR",
    "CVDR1HR",
    "CVDR2HR",
    "CVDR3HR",
    "CVDESVO2",
    "CVDFITLV"
]

for name, df in [
    ("1999-2000", cvx_1999),
    ("2001-2002", cvx_2001),
    ("2003-2004", cvx_2003),
    ("combined cvx", cvx)
]:
    print(
        name,
        [col for col in cvx_target if col in df.columns]
    )

print("\nCombined CVX rows:", len(cvx))
print("Unique SEQN:", cvx["SEQN"].nunique())
print("Duplicate SEQN:", cvx["SEQN"].duplicated().sum())

print("\nExisting cvx_features columns:")
print(cvx_features.columns.tolist())

1999-2000 ['SEQN', 'CVDVOMAX', 'CVDWHR', 'CVDR1HR', 'CVDR2HR', 'CVDR3HR', 'CVDESVO2', 'CVDFITLV']
2001-2002 ['SEQN', 'CVDVOMAX', 'CVDWHR', 'CVDR1HR', 'CVDR2HR', 'CVDR3HR', 'CVDESVO2', 'CVDFITLV']
2003-2004 ['SEQN', 'CVDVOMAX', 'CVDWHR', 'CVDR1HR', 'CVDR2HR', 'CVDR3HR', 'CVDESVO2', 'CVDFITLV']
combined cvx ['SEQN', 'CVDVOMAX', 'CVDWHR', 'CVDR1HR', 'CVDR2HR', 'CVDR3HR', 'CVDESVO2', 'CVDFITLV']

Combined CVX rows: 14417
Unique SEQN: 14417
Duplicate SEQN: 0

Existing cvx_features columns:
['SEQN', 'CVDVOMAX', 'CVDESVO2', 'CVDFITLV', 'CVDR1HR', 'CVDR2HR', 'CVDR3HR']


In [398]:
cvx_cols = [
    "SEQN",
    "CVDVOMAX",
    "CVDWHR",
    "CVDR1HR",
    "CVDR2HR",
    "CVDR3HR",
    "CVDESVO2",
    "CVDFITLV"
]

cvx_features = cvx[cvx_cols].copy()

print("CVX feature rows:", len(cvx_features))
print("Unique SEQN:", cvx_features["SEQN"].nunique())
print("Duplicate SEQN:", cvx_features["SEQN"].duplicated().sum())

CVX feature rows: 14417
Unique SEQN: 14417
Duplicate SEQN: 0


In [399]:
cvx_vars = [
    "CVDVOMAX",
    "CVDWHR",
    "CVDR1HR",
    "CVDR2HR",
    "CVDR3HR",
    "CVDESVO2",
    "CVDFITLV"
]

# Remove old CVX copies if this notebook has been rerun
cols_to_drop = []

for col in cvx_vars:
    for version in [col, f"{col}_x", f"{col}_y"]:
        if version in cohort.columns:
            cols_to_drop.append(version)

cohort = cohort.drop(columns=cols_to_drop)

# Merge clean CVX features
cohort = cohort.merge(
    cvx_features,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))

for col in cvx_vars:
    print(
        col,
        "available:",
        cohort[col].notna().sum(),
        "| missing:",
        cohort[col].isna().sum()
    )

Cohort rows: 9957
CVDVOMAX available: 1140 | missing: 8817
CVDWHR available: 922 | missing: 9035
CVDR1HR available: 950 | missing: 9007
CVDR2HR available: 995 | missing: 8962
CVDR3HR available: 471 | missing: 9486
CVDESVO2 available: 907 | missing: 9050
CVDFITLV available: 907 | missing: 9050


In [400]:
cvx_any = cohort[cvx_vars].notna().any(axis=1)

print("Any CVX available:", cvx_any.sum())

print(
    "CVX available age 40-49:",
    cvx_any[cohort["RIDAGEYR"].between(40, 49)].sum()
)

print(
    "CVX available age 50+:",
    cvx_any[cohort["RIDAGEYR"] >= 50].sum()
)

print("\nBy NHANES cycle:")
print(
    pd.crosstab(
        cohort["SDDSRVYR"],
        cvx_any,
        margins=True
    )
)

Any CVX available: 1140
CVX available age 40-49: 1140
CVX available age 50+: 0

By NHANES cycle:
col_0     False  True   All
SDDSRVYR                   
1.0        2814   368  3182
2.0        3050   433  3483
3.0        2953   339  3292
All        8817  1140  9957


In [401]:
print([
    name for name in globals()
    if "lab10" in name.lower() or "gh" in name.lower()
])

['gh_1999', 'gh_2001', 'gh_2003']


In [402]:
print([
    name for name in globals()
    if "gly" in name.lower()
])

['glyco']


In [403]:
from pathlib import Path

data_dir = Path("../data")

matches = [
    p.name for p in data_dir.rglob("*")
    if "lab10" in p.name.lower()
]

print(matches)

['LAB10.xpt', 'LAB10AM.xpt']


In [404]:
from pathlib import Path
import pandas as pd

data_dir = Path("../data/raw")

gh_1999 = pd.read_sas(data_dir / "LAB10.xpt", format="xport")
gh_2001 = pd.read_sas(data_dir / "L10_B.xpt", format="xport")
gh_2003 = pd.read_sas(data_dir / "L10_C.xpt", format="xport")

print("1999-2000:", gh_1999[["SEQN", "LBXGH"]].shape)
print("2001-2002:", gh_2001[["SEQN", "LBXGH"]].shape)
print("2003-2004:", gh_2003[["SEQN", "LBXGH"]].shape)

1999-2000: (6758, 2)
2001-2002: (7445, 2)
2003-2004: (6990, 2)


In [405]:
glyco = pd.concat(
    [
        gh_1999[["SEQN", "LBXGH"]],
        gh_2001[["SEQN", "LBXGH"]],
        gh_2003[["SEQN", "LBXGH"]],
    ],
    ignore_index=True
)

print("Glyco rows:", len(glyco))
print("Unique SEQN:", glyco["SEQN"].nunique())
print("Duplicate SEQN:", glyco["SEQN"].duplicated().sum())

Glyco rows: 21193
Unique SEQN: 21193
Duplicate SEQN: 0


In [406]:
# Remove old copies if notebook was rerun
for col in ["LBXGH", "LBXGH_x", "LBXGH_y"]:
    if col in cohort.columns:
        cohort = cohort.drop(columns=[col])

cohort = cohort.merge(
    glyco,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))
print("A1c available:", cohort["LBXGH"].notna().sum())
print("A1c missing:", cohort["LBXGH"].isna().sum())

print("\nA1c summary:")
print(cohort["LBXGH"].describe())

Cohort rows: 9957
A1c available: 8719
A1c missing: 1238

A1c summary:
count    8719.000000
mean        5.808028
std         1.164403
min         3.300000
25%         5.200000
50%         5.500000
75%         5.900000
max        18.800000
Name: LBXGH, dtype: float64


In [407]:
print([
    name for name in globals()
    if "lab18" in name.lower()
])

[]


In [408]:
print([
    p.name for p in data_dir.rglob("*")
    if "18" in p.name.lower()
])

['LAB18.xpt', 'LAB18T4.xpt']


In [409]:
chem_1999 = pd.read_sas(data_dir / "LAB18.xpt", format="xport")
chem_2001 = pd.read_sas(data_dir / "L40_B.xpt", format="xport")
chem_2003 = pd.read_sas(data_dir / "L40_C.xpt", format="xport")

chem_targets = [
    "SEQN",
    "LBXSPH",    # phosphorus
    "LBXSTB",    # bilirubin
    "LBXSTP",    # total protein
    "LBXSTR",    # triglyceride
    "LBXSUA",    # uric acid
    "LBXSNASI",  # sodium
    "LBXSKSI",   # potassium
    "LBXSCLSI",  # chloride
    "LBXSOSSI",  # osmolality
    "LBXSGB"     # globulin
]

for name, df in [
    ("1999-2000", chem_1999),
    ("2001-2002", chem_2001),
    ("2003-2004", chem_2003),
]:
    print("\n", name)
    print("Rows:", len(df))
    print("Found:", [c for c in chem_targets if c in df.columns])
    print("Missing:", [c for c in chem_targets if c not in df.columns])


 1999-2000
Rows: 6758
Found: ['SEQN', 'LBXSPH', 'LBXSTB', 'LBXSTP', 'LBXSTR', 'LBXSUA', 'LBXSNASI', 'LBXSKSI', 'LBXSCLSI', 'LBXSOSSI', 'LBXSGB']
Missing: []

 2001-2002
Rows: 7445
Found: ['SEQN', 'LBXSTP', 'LBXSTR', 'LBXSUA', 'LBXSNASI', 'LBXSKSI', 'LBXSCLSI', 'LBXSOSSI', 'LBXSGB']
Missing: ['LBXSPH', 'LBXSTB']

 2003-2004
Rows: 6990
Found: ['SEQN', 'LBXSPH', 'LBXSTB', 'LBXSTP', 'LBXSTR', 'LBXSUA', 'LBXSNASI', 'LBXSKSI', 'LBXSCLSI', 'LBXSOSSI', 'LBXSGB']
Missing: []


In [410]:
print("\n1999 columns containing relevant keywords:")
print([
    c for c in chem_1999.columns
    if any(x in c for x in ["SPH", "STB", "STP", "STR", "SUA", "SNA", "SK", "SCL", "SOS", "SGB"])
])

print("\n2001 columns containing relevant keywords:")
print([
    c for c in chem_2001.columns
    if any(x in c for x in ["SPH", "STB", "STP", "STR", "SUA", "SNA", "SK", "SCL", "SOS", "SGB"])
])

print("\n2003 columns containing relevant keywords:")
print([
    c for c in chem_2003.columns
    if any(x in c for x in ["SPH", "STB", "STP", "STR", "SUA", "SNA", "SK", "SCL", "SOS", "SGB"])
])


1999 columns containing relevant keywords:
['LBXSPH', 'LBDSPHSI', 'LBXSTB', 'LBDSTBSI', 'LBXSTP', 'LBDSTPSI', 'LBXSTR', 'LBDSTRSI', 'LBXSUA', 'LBDSUASI', 'LBXSNASI', 'LBXSKSI', 'LBXSCLSI', 'LBXSOSSI', 'LBXSGB', 'LBDSGBSI']

2001 columns containing relevant keywords:
['LBDSPH', 'LBDSPHSI', 'LBDSTB', 'LBDSTBSI', 'LBXSTP', 'LBDSTPSI', 'LBXSTR', 'LBDSTRSI', 'LBXSUA', 'LBDSUASI', 'LBXSNASI', 'LBXSKSI', 'LBXSCLSI', 'LBXSOSSI', 'LBXSGB', 'LBDSGBSI']

2003 columns containing relevant keywords:
['LBXSPH', 'LBDSPHSI', 'LBXSTB', 'LBDSTBSI', 'LBXSTP', 'LBDSTPSI', 'LBXSTR', 'LBDSTRSI', 'LBXSUA', 'LBDSUASI', 'LBXSNASI', 'LBXSKSI', 'LBXSCLSI', 'LBXSOSSI', 'LBXSGB', 'LBDSGBSI']


In [411]:
chem_cols = [
    "SEQN",
    "LBXSPH",
    "LBXSTB",
    "LBXSTP",
    "LBXSTR",
    "LBXSUA",
    "LBXSNASI",
    "LBXSKSI",
    "LBXSCLSI",
    "LBXSOSSI",
    "LBXSGB"
]

# 1999-2000
chem_1999_clean = chem_1999[chem_cols].copy()

# 2001-2002: harmonize two renamed variables
chem_2001_clean = chem_2001[
    [
        "SEQN",
        "LBDSPH",
        "LBDSTB",
        "LBXSTP",
        "LBXSTR",
        "LBXSUA",
        "LBXSNASI",
        "LBXSKSI",
        "LBXSCLSI",
        "LBXSOSSI",
        "LBXSGB"
    ]
].copy()

chem_2001_clean = chem_2001_clean.rename(
    columns={
        "LBDSPH": "LBXSPH",
        "LBDSTB": "LBXSTB"
    }
)

# 2003-2004
chem_2003_clean = chem_2003[chem_cols].copy()

# Combine cycles
chemistry = pd.concat(
    [
        chem_1999_clean,
        chem_2001_clean,
        chem_2003_clean
    ],
    ignore_index=True
)

print("Chemistry rows:", len(chemistry))
print("Unique SEQN:", chemistry["SEQN"].nunique())
print("Duplicate SEQN:", chemistry["SEQN"].duplicated().sum())

Chemistry rows: 21193
Unique SEQN: 21193
Duplicate SEQN: 0


In [412]:
chem_vars = chem_cols[1:]

# Remove old copies if notebook was rerun
cols_to_drop = []

for col in chem_vars:
    for version in [col, f"{col}_x", f"{col}_y"]:
        if version in cohort.columns:
            cols_to_drop.append(version)

cohort = cohort.drop(columns=cols_to_drop)

# Merge
cohort = cohort.merge(
    chemistry,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))

for col in chem_vars:
    print(
        col,
        "available:",
        cohort[col].notna().sum(),
        "| missing:",
        cohort[col].isna().sum()
    )

Cohort rows: 9957
LBXSPH available: 8541 | missing: 1416
LBXSTB available: 8540 | missing: 1417
LBXSTP available: 8538 | missing: 1419
LBXSTR available: 8542 | missing: 1415
LBXSUA available: 8541 | missing: 1416
LBXSNASI available: 8541 | missing: 1416
LBXSKSI available: 8541 | missing: 1416
LBXSCLSI available: 8538 | missing: 1419
LBXSOSSI available: 8541 | missing: 1416
LBXSGB available: 8538 | missing: 1419


In [413]:
print([
    name for name in globals()
    if "paq" in name.lower()
])

['paq_1999', 'paq_2001', 'paq_2003', 'paq', 'paq_cols', 'paq_features']


In [414]:
print([
    p.name for p in data_dir.rglob("*")
    if "paq" in p.name.lower()
])

['PAQ_B.xpt', 'PAQ_C.xpt', 'PAQ.xpt']


In [415]:
for name, df in [
    ("1999-2000", paq_1999),
    ("2001-2002", paq_2001),
    ("2003-2004", paq_2003),
    ("combined", paq),
]:
    print(
        name,
        [c for c in ["SEQN", "PAQ180"] if c in df.columns]
    )

print("\nPAQ rows:", len(paq))
print("Unique SEQN:", paq["SEQN"].nunique())
print("Duplicate SEQN:", paq["SEQN"].duplicated().sum())

print("\nExisting paq_features:")
print(paq_features.columns.tolist())

print("\nPAQ180 values:")
print(paq["PAQ180"].value_counts(dropna=False).sort_index())

1999-2000 ['SEQN', 'PAQ180']
2001-2002 ['SEQN', 'PAQ180']
2003-2004 ['SEQN', 'PAQ180']
combined ['SEQN', 'PAQ180']

PAQ rows: 28560
Unique SEQN: 28560
Duplicate SEQN: 0

Existing paq_features:
['SEQN', 'PAQ180']

PAQ180 values:
PAQ180
1.0    4824
2.0    9735
3.0    3079
4.0    1220
7.0      11
9.0      16
NaN    9675
Name: count, dtype: int64


In [416]:
# Clean PAQ180 nonresponse codes
paq_features = paq[["SEQN", "PAQ180"]].copy()

paq_features["PAQ180"] = paq_features["PAQ180"].replace(
    [7, 9],
    pd.NA
)

print(paq_features["PAQ180"].value_counts(dropna=False).sort_index())

PAQ180
1.0     4824
2.0     9735
3.0     3079
4.0     1220
NaN     9675
<NA>      27
Name: count, dtype: int64


In [417]:
# Remove old PAQ180 copies if notebook was rerun
for col in ["PAQ180", "PAQ180_x", "PAQ180_y"]:
    if col in cohort.columns:
        cohort = cohort.drop(columns=[col])

cohort = cohort.merge(
    paq_features,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))
print("PAQ180 available:", cohort["PAQ180"].notna().sum())
print("PAQ180 missing:", cohort["PAQ180"].isna().sum())

print("\nPAQ180 distribution:")
print(cohort["PAQ180"].value_counts(dropna=False).sort_index())

Cohort rows: 9957
PAQ180 available: 9928
PAQ180 missing: 29

PAQ180 distribution:
PAQ180
1.0     2981
2.0     5251
3.0     1235
4.0      461
<NA>      24
NaN        5
Name: count, dtype: int64


In [418]:
cohort["PAQ180"] = pd.to_numeric(
    cohort["PAQ180"],
    errors="coerce"
)

print(cohort["PAQ180"].value_counts(dropna=False).sort_index())

PAQ180
1.0    2981
2.0    5251
3.0    1235
4.0     461
NaN      29
Name: count, dtype: int64


In [419]:
print([
    name for name in globals()
    if "ssbnp" in name.lower() or "bnp" in name.lower()
])

print([
    p.name for p in data_dir.rglob("*")
    if "ssbnp" in p.name.lower() or "bnp" in p.name.lower()
])

['bnp', 'bnp_features']
['SSBNP_A.xpt']


In [420]:
print("BNP rows:", len(bnp))
print("Unique SEQN:", bnp["SEQN"].nunique())
print("Duplicate SEQN:", bnp["SEQN"].duplicated().sum())

print("\nBNP columns:")
print(bnp.columns.tolist())

print("\nExisting bnp_features:")
print(bnp_features.columns.tolist())

BNP rows: 21206
Unique SEQN: 21206
Duplicate SEQN: 0

BNP columns:
['SEQN', 'WTSSCB2Y', 'WTSSCB4Y', 'SSBNP', 'SSBNPL', 'SSPRIS']

Existing bnp_features:
['SEQN', 'SSBNP']


In [421]:
# Keep only the paper predictor
bnp_features = bnp[["SEQN", "SSBNP"]].copy()

# Remove old copies if notebook was rerun
for col in ["SSBNP", "SSBNP_x", "SSBNP_y"]:
    if col in cohort.columns:
        cohort = cohort.drop(columns=[col])

cohort = cohort.merge(
    bnp_features,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))
print("NT-proBNP available:", cohort["SSBNP"].notna().sum())
print("NT-proBNP missing:", cohort["SSBNP"].isna().sum())

print("\nNT-proBNP summary:")
print(cohort["SSBNP"].describe())

Cohort rows: 9957
NT-proBNP available: 7967
NT-proBNP missing: 1990

NT-proBNP summary:
count     7967.000000
mean       281.343135
std       1253.516937
min          3.540000
25%         34.495000
50%         74.880000
75%        180.650000
max      35000.000000
Name: SSBNP, dtype: float64


In [422]:
print([
    name for name in globals()
    if "trop" in name.lower() or "sstnt" in name.lower()
])

print([
    p.name for p in data_dir.rglob("*")
    if "trop" in p.name.lower() or "sstnt" in p.name.lower()
])

['troponin', 'troponin_features']
['SSTROP_A.xpt']


In [423]:
print("Troponin rows:", len(troponin))
print("Unique SEQN:", troponin["SEQN"].nunique())
print("Duplicate SEQN:", troponin["SEQN"].duplicated().sum())

print("\nTroponin columns:")
print(troponin.columns.tolist())

print("\nExisting troponin_features:")
print(troponin_features.columns.tolist())

Troponin rows: 21743
Unique SEQN: 21743
Duplicate SEQN: 0

Troponin columns:
['SEQN', 'WTSSCB2Y', 'WTSSCB4Y', 'SSTNT', 'SSTNTLC', 'SSTNIA', 'SSTNIALC', 'SSTNIS', 'SSTNISLC', 'SSTNIO', 'SSTNIOLC', 'SSPRISTP']

Existing troponin_features:
['SEQN', 'SSTNT']


In [424]:
troponin_features = troponin[["SEQN", "SSTNT"]].copy()

# Remove old copies if notebook was rerun
for col in ["SSTNT", "SSTNT_x", "SSTNT_y"]:
    if col in cohort.columns:
        cohort = cohort.drop(columns=[col])

cohort = cohort.merge(
    troponin_features,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))
print("Troponin-T available:", cohort["SSTNT"].notna().sum())
print("Troponin-T missing:", cohort["SSTNT"].isna().sum())

print("\nTroponin-T summary:")
print(cohort["SSTNT"].describe())

Cohort rows: 9957
Troponin-T available: 7966
Troponin-T missing: 1991

Troponin-T summary:
count    7.966000e+03
mean     1.110145e+01
std      1.542528e+01
min      5.397605e-79
25%      4.840000e+00
50%      7.230000e+00
75%      1.217000e+01
max      4.171000e+02
Name: SSTNT, dtype: float64


In [425]:
print([
    name for name in globals()
    if any(x in name.lower() for x in [
        "lab06", "cad", "lead", "mercur",
        "folate", "b12", "cot"
    ])
])

['cot_1999', 'cot_2001', 'cot_2003', 'cot', 'cot_cols', 'cot_features', 'lab06_1999', 'lab06_2001', 'cot_2003_file']


In [426]:
print([
    p.name for p in data_dir.rglob("*")
    if any(x in p.name.lower() for x in [
        "lab06", "cad", "lead", "mercur",
        "fol", "b12", "cot"
    ])
])

['L06COT_C.xpt', 'LAB06.xpt']


In [427]:
print([
    p.name for p in data_dir.rglob("*")
    if "l06" in p.name.lower()
])

['L06BMT_C.xpt', 'L06COT_C.xpt', 'L06_B.xpt', 'L06NB_C.xpt']


In [428]:
lab06_1999 = pd.read_sas(
    data_dir / "LAB06.xpt",
    format="xport"
)

lab06_2001 = pd.read_sas(
    data_dir / "L06_B.xpt",
    format="xport"
)

bmt_2003 = pd.read_sas(
    data_dir / "L06BMT_C.xpt",
    format="xport"
)

nb_2003 = pd.read_sas(
    data_dir / "L06NB_C.xpt",
    format="xport"
)

cot_2003_file = pd.read_sas(
    data_dir / "L06COT_C.xpt",
    format="xport"
)

In [429]:
targets = [
    "SEQN",
    "LBXBCD",   # cadmium
    "LBXBPB",   # lead
    "LBXRBF",   # RBC folate
    "LBXTHG",   # total mercury
    "LBXB12",   # vitamin B12
    "LBXCOT"    # cotinine
]

for name, df in [
    ("1999-2000 LAB06", lab06_1999),
    ("2001-2002 L06_B", lab06_2001),
    ("2003-2004 metals", bmt_2003),
    ("2003-2004 nutrition", nb_2003),
    ("2003-2004 cotinine", cot_2003_file),
]:
    print("\n", name)
    print("Rows:", len(df))
    print("Exact targets found:")
    print([c for c in targets if c in df.columns])



 1999-2000 LAB06
Rows: 8832
Exact targets found:
['SEQN', 'LBXBCD', 'LBXBPB', 'LBXRBF', 'LBXTHG', 'LBXB12', 'LBXCOT']

 2001-2002 L06_B
Rows: 9929
Exact targets found:
['SEQN', 'LBXBCD', 'LBXBPB', 'LBXRBF', 'LBXTHG', 'LBXB12', 'LBXCOT']

 2003-2004 metals
Rows: 9179
Exact targets found:
['SEQN', 'LBXBCD', 'LBXBPB', 'LBXTHG']

 2003-2004 nutrition
Rows: 9179
Exact targets found:
['SEQN', 'LBXRBF', 'LBXB12']

 2003-2004 cotinine
Rows: 8556
Exact targets found:
['SEQN', 'LBXCOT']


In [430]:
keywords = [
    "BCD",   # cadmium
    "BPB",   # lead
    "RBF",   # RBC folate
    "THG",   # mercury
    "B12",   # vitamin B12
    "COT"    # cotinine
]

for name, df in [
    ("1999-2000 LAB06", lab06_1999),
    ("2001-2002 L06_B", lab06_2001),
    ("2003-2004 metals", bmt_2003),
    ("2003-2004 nutrition", nb_2003),
    ("2003-2004 cotinine", cot_2003_file),
]:
    print("\n", name)
    print([
        c for c in df.columns
        if any(k in c.upper() for k in keywords)
    ])


 1999-2000 LAB06
['LBXBPB', 'LBDBPBSI', 'LBXBCD', 'LBDBCDSI', 'LBXB12', 'LBDB12SI', 'LBXTHG', 'LBDTHGSI', 'LBXRBF', 'LBDRBFSI', 'LBXCOT', 'LBDCOTSI']

 2001-2002 L06_B
['LBXBCD', 'LBDBCDSI', 'LBXBPB', 'LBDBPBSI', 'LBXRBF', 'LBDRBFSI', 'LBXTHG', 'LBDTHGSI', 'LBXB12', 'LBDB12SI', 'LBXCOT', 'LBDCOTLC']

 2003-2004 metals
['LBXBCD', 'LBDBCDSI', 'LBXBPB', 'LBDBPBSI', 'LBXTHG', 'LBDTHGSI']

 2003-2004 nutrition
['LBXRBF', 'LBDRBFSI', 'LBXB12', 'LBDB12SI']

 2003-2004 cotinine
['LBXCOT']


In [431]:
trace_cols = [
    "SEQN",
    "LBXBCD",   # cadmium
    "LBXBPB",   # lead
    "LBXRBF",   # RBC folate
    "LBXTHG",   # mercury
    "LBXB12",   # vitamin B12
    "LBXCOT"    # cotinine
]

# 1999-2000
trace_1999 = lab06_1999[trace_cols].copy()

# 2001-2002
trace_2001 = lab06_2001[trace_cols].copy()

# 2003-2004: combine the three separate files
trace_2003 = (
    bmt_2003[
        ["SEQN", "LBXBCD", "LBXBPB", "LBXTHG"]
    ]
    .merge(
        nb_2003[
            ["SEQN", "LBXRBF", "LBXB12"]
        ],
        on="SEQN",
        how="outer",
        validate="one_to_one"
    )
    .merge(
        cot_2003_file[
            ["SEQN", "LBXCOT"]
        ],
        on="SEQN",
        how="outer",
        validate="one_to_one"
    )
)

print("1999 rows:", len(trace_1999))
print("2001 rows:", len(trace_2001))
print("2003 rows:", len(trace_2003))

print("2003 duplicate SEQN:",
      trace_2003["SEQN"].duplicated().sum())

1999 rows: 8832
2001 rows: 9929
2003 rows: 9179
2003 duplicate SEQN: 0


In [432]:
trace_elements = pd.concat(
    [
        trace_1999,
        trace_2001,
        trace_2003
    ],
    ignore_index=True
)

print("Trace-element rows:", len(trace_elements))
print("Unique SEQN:", trace_elements["SEQN"].nunique())
print(
    "Duplicate SEQN:",
    trace_elements["SEQN"].duplicated().sum()
)

Trace-element rows: 27940
Unique SEQN: 27940
Duplicate SEQN: 0


In [433]:
trace_vars = trace_cols[1:]

# Remove old copies from earlier notebook runs
cols_to_drop = []

for col in trace_vars:
    for version in [col, f"{col}_x", f"{col}_y"]:
        if version in cohort.columns:
            cols_to_drop.append(version)

cohort = cohort.drop(columns=cols_to_drop)

# Merge
cohort = cohort.merge(
    trace_elements,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))

for col in trace_vars:
    print(
        col,
        "available:",
        cohort[col].notna().sum(),
        "| missing:",
        cohort[col].isna().sum()
    )

Cohort rows: 9957
LBXBCD available: 8721 | missing: 1236
LBXBPB available: 8721 | missing: 1236
LBXRBF available: 8683 | missing: 1274
LBXTHG available: 3718 | missing: 6239
LBXB12 available: 8609 | missing: 1348
LBXCOT available: 8499 | missing: 1458


In [434]:
print([
    name for name in globals()
    if "lab25" in name.lower() or "cbc" in name.lower()
])

print([
    p.name for p in data_dir.rglob("*")
    if "25" in p.name.lower() or "cbc" in p.name.lower()
])

['cbc_1999', 'cbc_2001', 'cbc_2003', 'cbc', 'cbc_cols', 'cbc_features']
['LAB25.xpt', 'L25_B.xpt', 'L25_C.xpt']


In [435]:
cbc_targets = [
    "SEQN",
    "LBXWBCSI",
    "LBXLYPCT",
    "LBXMOPCT",
    "LBXNEPCT",
    "LBXEOPCT",
    "LBXBAPCT",
    "LBDLYMNO",
    "LBDMONO",
    "LBDNENO",
    "LBDEONO",
    "LBDBANO",
    "LBXRBCSI",
    "LBXHGB",
    "LBXHCT",
    "LBXMCVSI",
    "LBXMCHSI",
    "LBXMC",
    "LBXRDW",
    "LBXPLTSI"
]

print("CBC rows:", len(cbc))
print("Unique SEQN:", cbc["SEQN"].nunique())
print("Duplicate SEQN:", cbc["SEQN"].duplicated().sum())

print("\nTargets found in combined CBC:")
print([c for c in cbc_targets if c in cbc.columns])

print("\nTargets missing from combined CBC:")
print([c for c in cbc_targets if c not in cbc.columns])

print("\nExisting cbc_features columns:")
print(cbc_features.columns.tolist())

CBC rows: 27940
Unique SEQN: 27940
Duplicate SEQN: 0

Targets found in combined CBC:
['SEQN', 'LBXWBCSI', 'LBXLYPCT', 'LBXMOPCT', 'LBXNEPCT', 'LBXEOPCT', 'LBXBAPCT', 'LBDLYMNO', 'LBDMONO', 'LBDNENO', 'LBDEONO', 'LBDBANO', 'LBXRBCSI', 'LBXHGB', 'LBXHCT', 'LBXMCVSI', 'LBXMCHSI', 'LBXMC', 'LBXRDW', 'LBXPLTSI']

Targets missing from combined CBC:
[]

Existing cbc_features columns:
['SEQN', 'LBXWBCSI', 'LBXLYPCT', 'LBXMOPCT', 'LBXNEPCT', 'LBXEOPCT', 'LBXBAPCT', 'LBDLYMNO', 'LBDMONO', 'LBDNENO', 'LBDEONO', 'LBDBANO', 'LBXRBCSI', 'LBXHGB', 'LBXHCT', 'LBXMCVSI', 'LBXMCHSI', 'LBXMC', 'LBXRDW', 'LBXPLTSI']


In [436]:
cbc_features = cbc[cbc_targets].copy()

cbc_vars = cbc_targets[1:]

# Remove old copies if notebook was rerun
cols_to_drop = []

for col in cbc_vars:
    for version in [col, f"{col}_x", f"{col}_y"]:
        if version in cohort.columns:
            cols_to_drop.append(version)

cohort = cohort.drop(columns=cols_to_drop)

# Merge CBC block
cohort = cohort.merge(
    cbc_features,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))

for col in cbc_vars:
    print(
        col,
        "available:",
        cohort[col].notna().sum(),
        "| missing:",
        cohort[col].isna().sum()
    )

Cohort rows: 9957
LBXWBCSI available: 8731 | missing: 1226
LBXLYPCT available: 8690 | missing: 1267
LBXMOPCT available: 8690 | missing: 1267
LBXNEPCT available: 8690 | missing: 1267
LBXEOPCT available: 8690 | missing: 1267
LBXBAPCT available: 8690 | missing: 1267
LBDLYMNO available: 8690 | missing: 1267
LBDMONO available: 8690 | missing: 1267
LBDNENO available: 8690 | missing: 1267
LBDEONO available: 8690 | missing: 1267
LBDBANO available: 8690 | missing: 1267
LBXRBCSI available: 8731 | missing: 1226
LBXHGB available: 8731 | missing: 1226
LBXHCT available: 8731 | missing: 1226
LBXMCVSI available: 8731 | missing: 1226
LBXMCHSI available: 8731 | missing: 1226
LBXMC available: 8731 | missing: 1226
LBXRDW available: 8731 | missing: 1226
LBXPLTSI available: 8731 | missing: 1226


In [437]:
print([
    name for name in globals()
    if any(x in name.lower() for x in [
        "lab10am", "glucose", "insulin", "cpep"
    ])
])

print([
    p.name for p in data_dir.rglob("*")
    if any(x in p.name.lower() for x in [
        "10am", "glu", "ins", "cpep"
    ])
])

[]
['L10AM_C.xpt', 'L10AM_B.xpt', 'LAB10AM.xpt']


In [438]:
fast_1999 = pd.read_sas(
    data_dir / "LAB10AM.xpt",
    format="xport"
)

fast_2001 = pd.read_sas(
    data_dir / "L10AM_B.xpt",
    format="xport"
)

fast_2003 = pd.read_sas(
    data_dir / "L10AM_C.xpt",
    format="xport"
)

fast_targets = [
    "SEQN",
    "LBXGLU",   # plasma glucose
    "LBXCPSI",  # C-peptide
    "LBXIN"     # insulin
]

for name, df in [
    ("1999-2000", fast_1999),
    ("2001-2002", fast_2001),
    ("2003-2004", fast_2003),
]:
    print("\n", name)
    print("Rows:", len(df))
    print("Found:", [c for c in fast_targets if c in df.columns])
    print("Missing:", [c for c in fast_targets if c not in df.columns])


 1999-2000
Rows: 3267
Found: ['SEQN', 'LBXGLU', 'LBXCPSI', 'LBXIN']
Missing: []

 2001-2002
Rows: 3666
Found: ['SEQN', 'LBXGLU', 'LBXCPSI', 'LBXIN']
Missing: []

 2003-2004
Rows: 3356
Found: ['SEQN', 'LBXGLU', 'LBXCPSI', 'LBXIN']
Missing: []


In [439]:
keywords = ["GLU", "CPS", "INS"]

for name, df in [
    ("1999-2000", fast_1999),
    ("2001-2002", fast_2001),
    ("2003-2004", fast_2003),
]:
    print("\n", name)
    print([
        c for c in df.columns
        if any(k in c.upper() for k in keywords)
    ])


 1999-2000
['LBXGLU', 'LBXGLUSI', 'LBXCPSI', 'LBXINSI']

 2001-2002
['LBXGLU', 'LBXGLUSI', 'LBXCPSI', 'LBXINSI']

 2003-2004
['LBXGLU', 'LBDGLUSI', 'LBXCPSI', 'LBDINSI']


In [440]:
fast_cols = [
    "SEQN",
    "LBXGLU",
    "LBXCPSI",
    "LBXIN"
]

fast_1999_clean = fast_1999[fast_cols].copy()
fast_2001_clean = fast_2001[fast_cols].copy()
fast_2003_clean = fast_2003[fast_cols].copy()

fasting = pd.concat(
    [
        fast_1999_clean,
        fast_2001_clean,
        fast_2003_clean
    ],
    ignore_index=True
)

print("Fasting rows:", len(fasting))
print("Unique SEQN:", fasting["SEQN"].nunique())
print("Duplicate SEQN:", fasting["SEQN"].duplicated().sum())

Fasting rows: 10289
Unique SEQN: 10289
Duplicate SEQN: 0


In [441]:
fast_vars = fast_cols[1:]

# Remove old copies if notebook was rerun
cols_to_drop = []

for col in fast_vars:
    for version in [col, f"{col}_x", f"{col}_y"]:
        if version in cohort.columns:
            cols_to_drop.append(version)

cohort = cohort.drop(columns=cols_to_drop)

cohort = cohort.merge(
    fasting,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))

for col in fast_vars:
    print(
        col,
        "available:",
        cohort[col].notna().sum(),
        "| missing:",
        cohort[col].isna().sum()
    )

Cohort rows: 9957
LBXGLU available: 4258 | missing: 5699
LBXCPSI available: 4187 | missing: 5770
LBXIN available: 4188 | missing: 5769


In [442]:
print([
    name for name in globals()
    if any(x in name.lower() for x in [
        "lab11", "crp", "bap"
    ])
])

print([
    p.name for p in data_dir.rglob("*")
    if any(x in p.name.lower() for x in [
        "11", "crp", "bap"
    ])
])

['crp_1999', 'crp_2001', 'crp_2003', 'crp', 'crp_cols', 'crp_features']
['LAB11.xpt', 'L11_B.xpt', 'L11_C.xpt']


In [443]:
lab11_targets = [
    "SEQN",
    "LBXCRP",
    "LBXBAP"
]

for name, df in [
    ("1999-2000", crp_1999),
    ("2001-2002", crp_2001),
    ("2003-2004", crp_2003),
    ("combined", crp),
]:
    print(
        name,
        [c for c in lab11_targets if c in df.columns]
    )

print("\nCRP rows:", len(crp))
print("Unique SEQN:", crp["SEQN"].nunique())
print("Duplicate SEQN:", crp["SEQN"].duplicated().sum())

print("\nExisting crp_features:")
print(crp_features.columns.tolist())

1999-2000 ['SEQN', 'LBXCRP', 'LBXBAP']
2001-2002 ['SEQN', 'LBXCRP', 'LBXBAP']
2003-2004 ['SEQN', 'LBXCRP', 'LBXBAP']
combined ['SEQN', 'LBXCRP', 'LBXBAP']

CRP rows: 26785
Unique SEQN: 26785
Duplicate SEQN: 0

Existing crp_features:
['SEQN', 'LBXCRP', 'LBXBAP']


In [444]:
lab11_features = crp[[
    "SEQN",
    "LBXCRP",
    "LBXBAP"
]].copy()

lab11_vars = [
    "LBXCRP",
    "LBXBAP"
]

# Remove old copies if notebook was rerun
cols_to_drop = []

for col in lab11_vars:
    for version in [col, f"{col}_x", f"{col}_y"]:
        if version in cohort.columns:
            cols_to_drop.append(version)

cohort = cohort.drop(columns=cols_to_drop)

# Merge
cohort = cohort.merge(
    lab11_features,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))

for col in lab11_vars:
    print(
        col,
        "available:",
        cohort[col].notna().sum(),
        "| missing:",
        cohort[col].isna().sum()
    )

Cohort rows: 9957
LBXCRP available: 8607 | missing: 1350
LBXBAP available: 6369 | missing: 3588


In [445]:
print([
    name for name in globals()
    if any(x in name.lower() for x in [
        "lab13", "chol", "hdl"
    ])
])

print([
    p.name for p in data_dir.rglob("*")
    if any(x in p.name.lower() for x in [
        "13", "chol", "hdl"
    ])
])

['hdl_1999', 'hdl_2001', 'hdl_2003', 'hdl', 'hdl_cols', 'hdl_features']
['L13AM_C.xpt', 'L13AM_B.xpt', 'L13_C.xpt', 'LAB13.xpt', 'L13_B.xpt', 'LAB13AM.xpt']


In [446]:
chol_targets = [
    "SEQN",
    "LBXTC",
    "LBDHDL"
]

for name, df in [
    ("1999-2000", hdl_1999),
    ("2001-2002", hdl_2001),
    ("2003-2004", hdl_2003),
    ("combined", hdl),
]:
    print(
        name,
        [c for c in chol_targets if c in df.columns]
    )

print("\nRows:", len(hdl))
print("Unique SEQN:", hdl["SEQN"].nunique())
print("Duplicate SEQN:", hdl["SEQN"].duplicated().sum())

print("\nExisting hdl_features:")
print(hdl_features.columns.tolist())

1999-2000 ['SEQN', 'LBXTC', 'LBDHDL']
2001-2002 ['SEQN', 'LBXTC', 'LBDHDL']
2003-2004 ['SEQN', 'LBXTC', 'LBDHDL']
combined ['SEQN', 'LBXTC', 'LBDHDL']

Rows: 26162
Unique SEQN: 26162
Duplicate SEQN: 0

Existing hdl_features:
['SEQN', 'LBDHDL']


In [447]:
chol_features = hdl[[
    "SEQN",
    "LBXTC",
    "LBDHDL"
]].copy()

chol_vars = [
    "LBXTC",
    "LBDHDL"
]

# Remove old copies if notebook was rerun
cols_to_drop = []

for col in chol_vars:
    for version in [col, f"{col}_x", f"{col}_y"]:
        if version in cohort.columns:
            cols_to_drop.append(version)

cohort = cohort.drop(columns=cols_to_drop)

cohort = cohort.merge(
    chol_features,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))

for col in chol_vars:
    print(
        col,
        "available:",
        cohort[col].notna().sum(),
        "| missing:",
        cohort[col].isna().sum()
    )

Cohort rows: 9957
LBXTC available: 8551 | missing: 1406
LBDHDL available: 8549 | missing: 1408


In [448]:
chem2_targets = [
    "SEQN",
    "LBXSAL",     # albumin
    "LBXSATSI",   # ALT
    "LBXSASSI",   # AST
    "LBXSAPSI",   # alkaline phosphatase
    "LBXSBU",     # BUN
    "LBXSCA",     # calcium
    "LBXSCH",     # total cholesterol, chemistry profile
    "LBXSGTSI",   # gamma-GT
    "LBXSGL",     # serum glucose
    "LBXSIR",     # iron
    "LBXSLDSI"    # LDH
]

for name, df in [
    ("1999-2000", chem_1999),
    ("2001-2002", chem_2001),
    ("2003-2004", chem_2003),
]:
    print("\n", name)
    print("Found:", [c for c in chem2_targets if c in df.columns])
    print("Missing:", [c for c in chem2_targets if c not in df.columns])


 1999-2000
Found: ['SEQN', 'LBXSAL', 'LBXSATSI', 'LBXSASSI', 'LBXSAPSI', 'LBXSBU', 'LBXSCA', 'LBXSCH', 'LBXSGTSI', 'LBXSGL', 'LBXSIR', 'LBXSLDSI']
Missing: []

 2001-2002
Found: ['SEQN', 'LBXSAL', 'LBXSATSI', 'LBXSASSI', 'LBXSBU', 'LBXSCA', 'LBXSCH', 'LBXSGTSI', 'LBXSGL', 'LBXSIR']
Missing: ['LBXSAPSI', 'LBXSLDSI']

 2003-2004
Found: ['SEQN', 'LBXSAL', 'LBXSATSI', 'LBXSASSI', 'LBXSAPSI', 'LBXSBU', 'LBXSCA', 'LBXSCH', 'LBXSGTSI', 'LBXSGL', 'LBXSIR', 'LBXSLDSI']
Missing: []


In [449]:
chem2_cols = [
    "SEQN",
    "LBXSAL",
    "LBXSATSI",
    "LBXSASSI",
    "LBXSAPSI",
    "LBXSBU",
    "LBXSCA",
    "LBXSCH",
    "LBXSGTSI",
    "LBXSGL",
    "LBXSIR",
    "LBXSLDSI"
]

# 1999-2000
chem2_1999 = chem_1999[chem2_cols].copy()

# 2001-2002: harmonize two renamed variables
chem2_2001 = chem_2001[
    [
        "SEQN",
        "LBXSAL",
        "LBXSATSI",
        "LBXSASSI",
        "LBDSAPSI",
        "LBXSBU",
        "LBXSCA",
        "LBXSCH",
        "LBXSGTSI",
        "LBXSGL",
        "LBXSIR",
        "LBDSLDSI"
    ]
].copy()

chem2_2001 = chem2_2001.rename(
    columns={
        "LBDSAPSI": "LBXSAPSI",
        "LBDSLDSI": "LBXSLDSI"
    }
)

# 2003-2004
chem2_2003 = chem_2003[chem2_cols].copy()

# Combine cycles
chemistry2 = pd.concat(
    [
        chem2_1999,
        chem2_2001,
        chem2_2003
    ],
    ignore_index=True
)

print("Chemistry-2 rows:", len(chemistry2))
print("Unique SEQN:", chemistry2["SEQN"].nunique())
print(
    "Duplicate SEQN:",
    chemistry2["SEQN"].duplicated().sum()
)

Chemistry-2 rows: 21193
Unique SEQN: 21193
Duplicate SEQN: 0


In [450]:
chem2_vars = chem2_cols[1:]

# Remove old copies if notebook was rerun
cols_to_drop = []

for col in chem2_vars:
    for version in [col, f"{col}_x", f"{col}_y"]:
        if version in cohort.columns:
            cols_to_drop.append(version)

cohort = cohort.drop(columns=cols_to_drop)

# Merge
cohort = cohort.merge(
    chemistry2,
    on="SEQN",
    how="left",
    validate="many_to_one"
)

print("Cohort rows:", len(cohort))

for col in chem2_vars:
    print(
        col,
        "available:",
        cohort[col].notna().sum(),
        "| missing:",
        cohort[col].isna().sum()
    )

Cohort rows: 9957
LBXSAL available: 8541 | missing: 1416
LBXSATSI available: 8541 | missing: 1416
LBXSASSI available: 8541 | missing: 1416
LBXSAPSI available: 8541 | missing: 1416
LBXSBU available: 8541 | missing: 1416
LBXSCA available: 8541 | missing: 1416
LBXSCH available: 8543 | missing: 1414
LBXSGTSI available: 8541 | missing: 1416
LBXSGL available: 8541 | missing: 1416
LBXSIR available: 8540 | missing: 1417
LBXSLDSI available: 8539 | missing: 1418


In [451]:
# Save current feature-building checkpoint
cohort.to_pickle("../results/cohort_features_checkpoint.pkl")

print("Saved:", cohort.shape)

Saved: (9957, 265)


In [1]:
import pandas as pd
from pathlib import Path

files_to_check = [
    "dnmepi.sas7bdat",
    "RXQ_RX.xpt",
    "RXQ_RX_B.xpt",
    "RXQ_RX_C.xpt"
]

for filename in files_to_check:
    filepath = Path("../data/raw") / filename

    print("\nFile:", filename)

    try:
        data = pd.read_sas(filepath)

        print("Rows:", data.shape[0])
        print("Columns:", data.shape[1])
        print("Variable names:")
        print(data.columns.tolist())

        if "SEQN" in data.columns:
            print("Unique participants:", data["SEQN"].nunique())
            print("Duplicate SEQN:", data["SEQN"].duplicated().sum())

    except Exception as e:
        print("Error reading file:", e)


File: dnmepi.sas7bdat
Rows: 4449
Columns: 32
Variable names:
['SEQN', 'XY_Estimation', 'HorvathAge', 'HannumAge', 'SkinBloodAge', 'PhenoAge', 'GDF15Mort', 'B2MMort', 'CystatinCMort', 'TIMP1Mort', 'ADMMort', 'PAI1Mort', 'LeptinMort', 'PACKYRSMort', 'CRPMort', 'logA1CMort', 'GrimAgeMort', 'GrimAge2Mort', 'HorvathTelo', 'YangCell', 'ZhangAge', 'LinAge', 'WeidnerAge', 'VidalBraloAge', 'DunedinPoAm', 'CD8TPP', 'CD4TPP', 'Nkcell', 'Bcell', 'MonoPP', 'NeuPP', 'WTDN4YR']
Unique participants: 4449
Duplicate SEQN: 0

File: RXQ_RX.xpt
Rows: 15643
Columns: 15
Variable names:
['SEQN', 'RXD030', 'RXD240B', 'NHCODE', 'RXQ250', 'RXD260', 'FDACODE1', 'FDACODE2', 'FDACODE3', 'FDACODE4', 'FDACODE5', 'FDACODE6', 'RXD295', 'RXQ280', 'RXDDRGID']
Unique participants: 9965
Duplicate SEQN: 5678

File: RXQ_RX_B.xpt
Rows: 18418
Columns: 14
Variable names:
['SEQN', 'RXD030', 'RXD240B', 'NHCODE', 'RXQ250', 'RXD260', 'FDACODE1', 'FDACODE2', 'FDACODE3', 'FDACODE4', 'FDACODE5', 'FDACODE6', 'RXD295', 'RXDDRGID']
Uniq

In [2]:
import pandas as pd
from pathlib import Path

# Load the saved NHANES cohort and DNA methylation data
base = Path("..") if Path("../results/cohort_features_checkpoint.pkl").exists() else Path(".")

cohort_check = pd.read_pickle(
    base / "results/cohort_features_checkpoint.pkl"
)

dna = pd.read_sas(
    base / "data/raw/dnmepi.sas7bdat"
)

# Identify the candidate methylation features
dna_features = [
    c for c in dna.columns
    if c not in ["SEQN", "WTDN4YR"]
]

# Confirm one row per participant
assert cohort_check["SEQN"].is_unique
assert dna["SEQN"].is_unique

# Match DNA records to our existing cohort
matched = cohort_check[
    ["SEQN", "SDDSRVYR"]
].merge(
    dna[["SEQN"] + dna_features],
    on="SEQN",
    how="left",
    validate="one_to_one"
)

matched["Has_DNA_Data"] = (
    matched[dna_features].notna().any(axis=1)
)

print("Original cohort:", len(cohort_check))
print("Cohort after matching:", len(matched))
print("Candidate DNA features:", len(dna_features))
print("Participants with DNA data:",
      matched["Has_DNA_Data"].sum())

print("\nDNA data availability by survey cycle:")
display(
    matched.groupby("SDDSRVYR")["Has_DNA_Data"]
    .agg(["count", "sum", "mean"])
    .round(3)
)

print("\nDNA variables:")
print(dna_features)

Original cohort: 9957
Cohort after matching: 9957
Candidate DNA features: 30
Participants with DNA data: 2532

DNA data availability by survey cycle:


,count,sum,mean
SDDSRVYR,,,
1.0,3182,1220,0.383
2.0,3483,1312,0.377
3.0,3292,0,0.000



DNA variables:
['XY_Estimation', 'HorvathAge', 'HannumAge', 'SkinBloodAge', 'PhenoAge', 'GDF15Mort', 'B2MMort', 'CystatinCMort', 'TIMP1Mort', 'ADMMort', 'PAI1Mort', 'LeptinMort', 'PACKYRSMort', 'CRPMort', 'logA1CMort', 'GrimAgeMort', 'GrimAge2Mort', 'HorvathTelo', 'YangCell', 'ZhangAge', 'LinAge', 'WeidnerAge', 'VidalBraloAge', 'DunedinPoAm', 'CD8TPP', 'CD4TPP', 'Nkcell', 'Bcell', 'MonoPP', 'NeuPP']


In [3]:
from pathlib import Path

# Verify no duplicate participants
assert cohort_check["SEQN"].is_unique
assert dna["SEQN"].is_unique
assert len(dna_features) == 30

# Make sure these columns aren't already present
overlap = set(dna_features) & set(cohort_check.columns)
assert not overlap, f"Already existing columns: {overlap}"

# Merge DNA measurements without dropping participants
cohort_with_dna = cohort_check.merge(
    dna[["SEQN"] + dna_features],
    on="SEQN",
    how="left",
    validate="one_to_one"
)

# Verify that the cohort remains unchanged in size
assert len(cohort_with_dna) == 9957
assert cohort_with_dna["SEQN"].is_unique

dna_available = cohort_with_dna[dna_features].notna().any(axis=1)

print("Participants:", len(cohort_with_dna))
print("Total dataset columns:", cohort_with_dna.shape[1])
print("DNA features added:", len(dna_features))
print("Participants with DNA data:", dna_available.sum())
print("Participants without DNA data:", (~dna_available).sum())

# Save a NEW checkpoint; preserve the previous dataset
output = Path("../results/cohort_features_with_dna_checkpoint.pkl")
output.parent.mkdir(parents=True, exist_ok=True)

cohort_with_dna.to_pickle(output)

print("\nSaved:", output)

Participants: 9957
Total dataset columns: 295
DNA features added: 30
Participants with DNA data: 2532
Participants without DNA data: 7425

Saved: ../results/cohort_features_with_dna_checkpoint.pkl


In [4]:
import pandas as pd
from pathlib import Path

base = Path("..") if Path("../data/raw/RXQ_RX.xpt").exists() else Path(".")

med_files = {
    "1999-2000": "RXQ_RX.xpt",
    "2001-2002": "RXQ_RX_B.xpt",
    "2003-2004": "RXQ_RX_C.xpt"
}

for cycle, filename in med_files.items():

    med = pd.read_sas(base / "data/raw" / filename)

    print(f"\n--- {cycle} ---")
    print("Medication records:", len(med))
    print("Unique participants:", med["SEQN"].nunique())
    print("Participants in our cohort:",
          med.loc[
              med["SEQN"].isin(cohort_with_dna["SEQN"]),
              "SEQN"
          ].nunique())

    print("\nColumns:")
    print(med.columns.tolist())

    # Show sample medication identifiers and names
    selected = [
        c for c in ["SEQN", "RXDDRGID", "RXDDRUG", "RXD030", "NHCODE"]
        if c in med.columns
    ]

    sample = med[selected].head(5).copy()

    for col in sample.columns:
        sample[col] = sample[col].map(
            lambda x: x.decode("utf-8", errors="replace")
            if isinstance(x, bytes) else x
        )

    display(sample)


--- 1999-2000 ---
Medication records: 15643
Unique participants: 9965
Participants in our cohort: 3182

Columns:
['SEQN', 'RXD030', 'RXD240B', 'NHCODE', 'RXQ250', 'RXD260', 'FDACODE1', 'FDACODE2', 'FDACODE3', 'FDACODE4', 'FDACODE5', 'FDACODE6', 'RXD295', 'RXQ280', 'RXDDRGID']


,SEQN,RXDDRGID,RXD030,NHCODE
0,1.0,d00088,1.0,03500
1,2.0,d00817,1.0,31700
2,2.0,d04037,1.0,56700
3,3.0,,2.0,
4,4.0,d03577,1.0,10200



--- 2001-2002 ---
Medication records: 18418
Unique participants: 11039
Participants in our cohort: 3483

Columns:
['SEQN', 'RXD030', 'RXD240B', 'NHCODE', 'RXQ250', 'RXD260', 'FDACODE1', 'FDACODE2', 'FDACODE3', 'FDACODE4', 'FDACODE5', 'FDACODE6', 'RXD295', 'RXDDRGID']


,SEQN,RXDDRGID,RXD030,NHCODE
0,9966.0,,2.0,
1,9967.0,,2.0,
2,9968.0,d03434,1.0,01200
3,9968.0,d04293,1.0,30400
4,9968.0,d04261,1.0,50900



--- 2003-2004 ---
Medication records: 18401
Unique participants: 10122
Participants in our cohort: 3292

Columns:
['SEQN', 'RXDUSE', 'RXDDRUG', 'RXDDRGID', 'RXQSEEN', 'RXDDAYS', 'RXDCOUNT']


,SEQN,RXDDRGID,RXDDRUG
0,21005.0,,
1,21006.0,,
2,21007.0,,
3,21008.0,,
4,21009.0,,


In [5]:
from pathlib import Path
import pandas as pd

path = Path("../data/raw/RXQ_DRUG.xpt")

drug_lookup = pd.read_sas(path)

print("Drug records:", len(drug_lookup))
print("Unique drug IDs:", drug_lookup["RXDDRGID"].nunique())
print("Duplicate drug IDs:",
      drug_lookup["RXDDRGID"].duplicated().sum())

print("\nTherapeutic classification columns:")
print([
    c for c in drug_lookup.columns
    if c.startswith(("RXDDCN", "RXDICN"))
])

print("\nExample drug information:")
display(
    drug_lookup[
        ["RXDDRGID", "RXDDRUG", "RXDDCN1A", "RXDDCN1B"]
    ].head(10)
)

Drug records: 1551
Unique drug IDs: 1551
Duplicate drug IDs: 0

Therapeutic classification columns:
['RXDDCN1A', 'RXDDCN1B', 'RXDDCN1C', 'RXDDCN2A', 'RXDDCN2B', 'RXDDCN2C', 'RXDDCN3A', 'RXDDCN3B', 'RXDDCN3C', 'RXDDCN4A', 'RXDDCN4B', 'RXDDCN4C', 'RXDICN1A', 'RXDICN1B', 'RXDICN1C', 'RXDICN2A', 'RXDICN2B', 'RXDICN2C', 'RXDICN3A', 'RXDICN3B', 'RXDICN3C', 'RXDICN4A', 'RXDICN4B', 'RXDICN4C', 'RXDICN5A', 'RXDICN5B', 'RXDICN5C', 'RXDICN6A', 'RXDICN6B', 'RXDICN6C']

Example drug information:


,RXDDRGID,RXDDRUG,RXDDCN1A,RXDDCN1B
0,b'a10061',b'BILE SALTS; PANCREATIN; PEPSIN',b'GASTROINTESTINAL AGENTS',b'MISCELLANEOUS GI AGENTS'
1,b'a10129',b'CALCIUM IODIDE; CODEINE',b'RESPIRATORY AGENTS',b'ANTITUSSIVES'
2,b'a10899',b'ESTRADIOL; ESTRIOL; ESTRONE',b'HORMONES/HORMONE MODIFIERS',b'SEX HORMONES'
3,b'a10900',b'ESTRADIOL; ESTRIOL',b'HORMONES/HORMONE MODIFIERS',b'SEX HORMONES'
4,b'a11077',b'HYDROCHLOROTHIAZIDE; LABETALOL',b'CARDIOVASCULAR AGENTS',b'ANTIHYPERTENSIVE COMBINATIONS'
5,b'a11122',b'ACETAMINOPHEN; CHLORZOXAZONE',b'CENTRAL NERVOUS SYSTEM AGENTS',b'MUSCLE RELAXANTS'
6,b'a11179',b'IODINATED GLYCEROL; THEOPHYLLINE',b'RESPIRATORY AGENTS',b'ANTIASTHMATIC COMBINATIONS'
7,b'a11273',b'DEXTROMETHORPHAN; IODINATED GLYCEROL',b'RESPIRATORY AGENTS',b'UPPER RESPIRATORY COMBINATIONS'
8,b'a11284',b'ETHINYL ESTRADIOL; FERROUS FUMARATE; NORETHI...,b'HORMONES/HORMONE MODIFIERS',b'SEX HORMONES'
9,b'a11457',b'CYANOCOBALAMIN; PYRIDOXINE; THIAMINE',b'NUTRITIONAL PRODUCTS',b'VITAMINS'


In [6]:
import pandas as pd
from pathlib import Path

raw_dir = Path("../data/raw")

lookup = pd.read_sas(raw_dir / "RXQ_DRUG.xpt")

# Convert SAS byte strings into ordinary text
def clean_ids(column):
    def clean(value):
        if pd.isna(value):
            return None
        if isinstance(value, bytes):
            value = value.decode("utf-8", errors="replace")
        return str(value).strip().lower() or None

    return column.map(clean)

lookup_ids = set(
    clean_ids(lookup["RXDDRGID"]).dropna()
)

med_files = {
    "1999-2000": "RXQ_RX.xpt",
    "2001-2002": "RXQ_RX_B.xpt",
    "2003-2004": "RXQ_RX_C.xpt"
}

results = []

for cycle, filename in med_files.items():
    med = pd.read_sas(raw_dir / filename)

    drug_ids = clean_ids(med["RXDDRGID"])

    has_id = drug_ids.notna()
    matched = drug_ids.isin(lookup_ids)

    results.append({
        "Cycle": cycle,
        "Records": len(med),
        "With_Drug_ID": int(has_id.sum()),
        "Matched_IDs": int(matched.sum()),
        "Match_Percent": round(
            100 * matched.sum() / has_id.sum(), 1
        ) if has_id.sum() else 0
    })

    unmatched = drug_ids[has_id & ~matched].unique()
    print(f"{cycle} unmatched ID examples:", unmatched[:10])

print("\nMedication lookup coverage:")
display(pd.DataFrame(results))

1999-2000 unmatched ID examples: <StringArray>
[]
Length: 0, dtype: str
2001-2002 unmatched ID examples: <StringArray>
[]
Length: 0, dtype: str
2003-2004 unmatched ID examples: <StringArray>
[]
Length: 0, dtype: str

Medication lookup coverage:


,Cycle,Records,With_Drug_ID,Matched_IDs,Match_Percent
0,1999-2000,15643,9138,9138,100.0
1,2001-2002,18418,11447,11447,100.0
2,2003-2004,18401,12231,12231,100.0


In [7]:
# Inspect therapeutic categories for our medication predictors

class_cols = [
    c for c in drug_lookup.columns
    if c.startswith(("RXDDCN", "RXDICN"))
]

# Decode SAS text values
categories = drug_lookup[class_cols].stack().map(
    lambda x: x.decode("utf-8", errors="replace")
    if isinstance(x, bytes) else str(x)
).str.strip()

search_terms = {
    "Cholesterol": r"LIPID|CHOLESTEROL|STATIN|HMG.CO.A",
    "Blood Pressure": (
        r"ANTIHYPERTENSIVE|DIURETIC|BETA.BLOCKER|"
        r"CALCIUM CHANNEL|ANGIOTENSIN|ACE INHIBITOR"
    ),
    "Diabetes": (
        r"ANTIDIABETIC|HYPOGLYCEMIC|INSULIN|BIGUANIDE"
    )
}

for group, pattern in search_terms.items():
    matches = categories[
        categories.str.contains(
            pattern, case=False, regex=True, na=False
        )
    ]

    print(f"\n--- {group} ---")
    print("Matching therapeutic category labels:")
    print(sorted(matches.unique().tolist()))


--- Cholesterol ---
Matching therapeutic category labels:
['ANTIHYPERLIPIDEMIC AGENTS', 'ANTIHYPERLIPIDEMIC COMBINATIONS', 'CHOLESTEROL ABSORPTION INHIBITORS', 'HMG-COA REDUCTASE INHIBITORS (STATINS)', 'MISCELLANEOUS ANTIHYPERLIPIDEMIC AGENTS', 'SOMATOSTATIN AND SOMATOSTATIN ANALOGS']

--- Blood Pressure ---
Matching therapeutic category labels:
['ACE INHIBITORS WITH CALCIUM CHANNEL BLOCKING AGENTS', 'ACE INHIBITORS WITH THIAZIDES', 'ANGIOTENSIN CONVERTING ENZYME (ACE) INHIBITORS', 'ANGIOTENSIN II INHIBITORS', 'ANGIOTENSIN II INHIBITORS WITH CALCIUM CHANNEL BLOCKERS', 'ANGIOTENSIN II INHIBITORS WITH THIAZIDES', 'ANGIOTENSIN RECEPTOR BLOCKERS AND NEPRILYSIN INHIBITORS', 'ANTIDIURETIC HORMONES', 'ANTIHYPERTENSIVE COMBINATIONS', 'BETA BLOCKERS WITH THIAZIDES', 'BETA BLOCKERS, CARDIOSELECTIVE', 'BETA BLOCKERS, NON-CARDIOSELECTIVE', 'CALCIUM CHANNEL BLOCKING AGENTS', 'DIURETICS', 'LOOP DIURETICS', 'MISCELLANEOUS ANTIHYPERTENSIVE COMBINATIONS', 'POTASSIUM SPARING DIURETICS WITH THIAZIDES', 

In [8]:
import pandas as pd

# CDC classification columns
class_cols = [
    c for c in drug_lookup.columns
    if c.startswith(("RXDDCN", "RXDICN"))
]

lookup_review = drug_lookup[
    ["RXDDRGID", "RXDDRUG"] + class_cols
].copy()

# Decode SAS strings
for col in lookup_review.columns:
    lookup_review[col] = lookup_review[col].map(
        lambda x: x.decode("utf-8").strip()
        if isinstance(x, bytes)
        else x
    )

# Convert classifications into a reviewable table
drug_classes = lookup_review.melt(
    id_vars=["RXDDRGID", "RXDDRUG"],
    value_vars=class_cols,
    var_name="Source",
    value_name="Category"
).dropna(subset=["Category"])

drug_classes = drug_classes[
    drug_classes["Category"].str.strip() != ""
]

# Avoid counting the same drug/category repeatedly
drug_classes = drug_classes.drop_duplicates(
    ["RXDDRGID", "Category"]
)

# Candidate therapeutic categories
patterns = {
    "Cholesterol": (
        r"ANTIHYPERLIPIDEMIC|CHOLESTEROL ABSORPTION|"
        r"HMG.COA REDUCTASE|FIBRIC ACID|BILE ACID"
    ),
    "Blood Pressure": (
        r"ANTIHYPERTENSIVE|ACE INHIBITOR|ANGIOTENSIN|"
        r"BETA.*BLOCK|CALCIUM CHANNEL BLOCK|DIURETIC|"
        r"RENIN INHIBITOR"
    ),
    "Diabetes": (
        r"ANTIDIABETIC|INSULIN|BIGUANIDE|SULFONYLUREA|"
        r"THIAZOLIDINEDIONE|ALPHA.GLU COSIDASE"
    )
}

for group, pattern in patterns.items():

    selected = drug_classes[
        drug_classes["Category"].str.contains(
            pattern, case=False, regex=True, na=False
        )
    ]

    summary = (
        selected.groupby("Category")
        .agg(
            Drugs=("RXDDRGID", "nunique"),
            Example=("RXDDRUG", "first")
        )
        .sort_values("Drugs", ascending=False)
    )

    print(f"\n=== {group} ===")
    print("Candidate categories:", len(summary))
    display(summary)


=== Cholesterol ===
Candidate categories: 7


,Drugs,Example
Category,,
ANTIHYPERLIPIDEMIC AGENTS,29,ANTIHYPERLIPIDEMIC AGENTS - UNSPECIFIED
HMG-COA REDUCTASE INHIBITORS (STATINS),14,HMG-COA REDUCTASE INHIBITORS - UNSPECIFIED
ANTIHYPERLIPIDEMIC COMBINATIONS,5,LOVASTATIN; NIACIN
MISCELLANEOUS ANTIHYPERLIPIDEMIC AGENTS,5,PROBUCOL
FIBRIC ACID DERIVATIVES,5,CLOFIBRATE
BILE ACID SEQUESTRANTS,3,CHOLESTYRAMINE
CHOLESTEROL ABSORPTION INHIBITORS,2,EZETIMIBE



=== Blood Pressure ===
Candidate categories: 21


,Drugs,Example
Category,,
DIURETICS,63,DIURETICS - UNSPECIFIED
ANTIHYPERTENSIVE COMBINATIONS,51,HYDROCHLOROTHIAZIDE; LABETALOL
THIAZIDE AND THIAZIDE-LIKE DIURETICS,51,THIAZIDE AND THIAZIDE-LIKE DIURETICS - UNSPECI...
ANGIOTENSIN II INHIBITORS,23,LOSARTAN
BETA-ADRENERGIC BLOCKING AGENTS,22,BETA-ADRENERGIC BLOCKING AGENTS - UNSPECIFIED
ANGIOTENSIN CONVERTING ENZYME (ACE) INHIBITORS,21,CAPTOPRIL
CALCIUM CHANNEL BLOCKING AGENTS,18,NIFEDIPINE
"BETA BLOCKERS, NON-CARDIOSELECTIVE",12,LABETALOL
"BETA BLOCKERS, CARDIOSELECTIVE",9,ATENOLOL



=== Diabetes ===
Candidate categories: 6


,Drugs,Example
Category,,
ANTIDIABETIC AGENTS,64,ANTIDIABETIC AGENTS - UNSPECIFIED
ANTIDIABETIC COMBINATIONS,17,GLYBURIDE; METFORMIN
INSULIN,17,INSULIN
SULFONYLUREAS,12,CHLORPROPAMIDE
BIGUANIDES,11,METFORMIN
THIAZOLIDINEDIONES,7,TROGLITAZONE


In [9]:
# Provisional therapeutic classes for our three medication predictors

cholesterol_classes = {
    "ANTIHYPERLIPIDEMIC AGENTS",
    "ANTIHYPERLIPIDEMIC COMBINATIONS",
    "MISCELLANEOUS ANTIHYPERLIPIDEMIC AGENTS",
    "HMG-COA REDUCTASE INHIBITORS (STATINS)",
    "FIBRIC ACID DERIVATIVES",
    "BILE ACID SEQUESTRANTS",
    "CHOLESTEROL ABSORPTION INHIBITORS",
}

blood_pressure_classes = {
    "DIURETICS",
    "ANTIHYPERTENSIVE COMBINATIONS",
    "THIAZIDE AND THIAZIDE-LIKE DIURETICS",
    "ANGIOTENSIN II INHIBITORS",
    "BETA-ADRENERGIC BLOCKING AGENTS",
    "ANGIOTENSIN CONVERTING ENZYME (ACE) INHIBITORS",
    "CALCIUM CHANNEL BLOCKING AGENTS",
    "BETA BLOCKERS, NON-CARDIOSELECTIVE",
    "BETA BLOCKERS, CARDIOSELECTIVE",
    "ANGIOTENSIN II INHIBITORS WITH THIAZIDES",
    "POTASSIUM-SPARING DIURETICS",
    "ACE INHIBITORS WITH THIAZIDES",
    "MISCELLANEOUS ANTIHYPERTENSIVE COMBINATIONS",
    "BETA BLOCKERS WITH THIAZIDES",
    "LOOP DIURETICS",
    "ANGIOTENSIN II INHIBITORS WITH CALCIUM CHANNEL BLOCKERS",
    "ACE INHIBITORS WITH CALCIUM CHANNEL BLOCKING AGENTS",
    "POTASSIUM SPARING DIURETICS WITH THIAZIDES",
    "RENIN INHIBITORS",
    "ANGIOTENSIN RECEPTOR BLOCKERS AND NEPRILYSIN INHIBITORS",
}

diabetes_classes = {
    "ANTIDIABETIC AGENTS",
    "ANTIDIABETIC COMBINATIONS",
    "INSULIN",
    "SULFONYLUREAS",
    "BIGUANIDES",
    "THIAZOLIDINEDIONES",
}

class_groups = {
    "DERIVED_HL_MED": cholesterol_classes,
    "DERIVED_HTN_MED": blood_pressure_classes,
    "DERIVED_DM_MED": diabetes_classes,
}

# Verify every selected category exists in our lookup
available_classes = set(drug_classes["Category"])

for name, selected_classes in class_groups.items():
    missing = selected_classes - available_classes
    assert not missing, f"{name}: unknown categories {missing}"

# Build one row per drug ID
drug_flags = lookup_review[
    ["RXDDRGID", "RXDDRUG"]
].drop_duplicates("RXDDRGID").copy()

for name, selected_classes in class_groups.items():
    matching_ids = set(
        drug_classes.loc[
            drug_classes["Category"].isin(selected_classes),
            "RXDDRGID"
        ]
    )

    drug_flags[name] = drug_flags["RXDDRGID"].isin(matching_ids)

# Verify unique drug IDs
assert drug_flags["RXDDRGID"].is_unique

print("Total drugs:", len(drug_flags))

print("\nDrug IDs classified by medication group:")
for name in class_groups:
    print(name, ":", int(drug_flags[name].sum()))

print("\nExample classified drugs:")
display(
    drug_flags[
        drug_flags[list(class_groups)].any(axis=1)
    ].head(15)
)

Total drugs: 1551

Drug IDs classified by medication group:
DERIVED_HL_MED : 29
DERIVED_HTN_MED : 118
DERIVED_DM_MED : 64

Example classified drugs:


,RXDDRGID,RXDDRUG,DERIVED_HL_MED,DERIVED_HTN_MED,DERIVED_DM_MED
4,a11077,HYDROCHLOROTHIAZIDE; LABETALOL,False,True,False
36,c00019,ANTIHYPERLIPIDEMIC AGENTS - UNSPECIFIED,True,False,False
40,c00047,BETA-ADRENERGIC BLOCKING AGENTS - UNSPECIFIED,False,True,False
41,c00049,DIURETICS - UNSPECIFIED,False,True,False
42,c00055,ANTIHYPERTENSIVE COMBINATIONS - UNSPECIFIED,False,True,False
56,c00099,ANTIDIABETIC AGENTS - UNSPECIFIED,False,False,True
72,c00156,THIAZIDE AND THIAZIDE-LIKE DIURETICS - UNSPECI...,False,True,False
77,c00173,HMG-COA REDUCTASE INHIBITORS - UNSPECIFIED,True,False,False
93,d00004,ATENOLOL,False,True,False
94,d00006,CAPTOPRIL,False,True,False


In [12]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# Build participant-level medication predictors
# ============================================================

raw_dir = Path("../data/raw")

flag_cols = [
    "DERIVED_HL_MED",
    "DERIVED_HTN_MED",
    "DERIVED_DM_MED"
]

# 1. Clean drug IDs consistently
def normalize_drug_id(series):

    def clean(value):
        if pd.isna(value):
            return None

        if isinstance(value, bytes):
            value = value.decode("utf-8", errors="replace")

        value = str(value).strip().lower()

        # Empty IDs must be treated as missing
        if value in ("", "nan", "none"):
            return None

        return value

    return series.map(clean)


# 2. Prepare the classified drug lookup
lookup_flags = drug_flags[
    ["RXDDRGID"] + flag_cols
].copy()

lookup_flags["RXDDRGID"] = normalize_drug_id(
    lookup_flags["RXDDRGID"]
)

assert lookup_flags["RXDDRGID"].notna().all()
assert lookup_flags["RXDDRGID"].is_unique


# 3. Identify medication files and use-response columns
med_files = {
    "1999-2000": ("RXQ_RX.xpt", "RXD030"),
    "2001-2002": ("RXQ_RX_B.xpt", "RXD030"),
    "2003-2004": ("RXQ_RX_C.xpt", "RXDUSE")
}

medication_records = []

for cycle, (filename, status_col) in med_files.items():

    print(f"\nProcessing {cycle}...")

    med = pd.read_sas(raw_dir / filename)

    records = med[
        ["SEQN", status_col, "RXDDRGID"]
    ].copy()

    records = records.rename(
        columns={status_col: "RX_USE"}
    )

    records["RXDDRGID"] = normalize_drug_id(
        records["RXDDRGID"]
    )

    records["Cycle"] = cycle

    original_rows = len(records)

    # Match each medication to its therapeutic classes
    records = records.merge(
        lookup_flags,
        on="RXDDRGID",
        how="left",
        validate="many_to_one"
    )

    assert len(records) == original_rows

    # Check whether any known drug IDs failed to match
    known_id = records["RXDDRGID"].notna()

    unmatched = records.loc[
        known_id & records[flag_cols].isna().any(axis=1),
        ["SEQN", "RXDDRGID"]
    ]

    print("Total records:", len(records))
    print("Records with drug IDs:", int(known_id.sum()))
    print("Unmatched drug IDs:", len(unmatched))

    if len(unmatched) > 0:
        display(unmatched.head(10))
        raise ValueError(
            f"Unmatched drug IDs found in {cycle}"
        )

    # An unknown drug ID is not evidence of a target medication
    # Preserve that uncertainty when deriving participant flags
    for col in flag_cols:
        records[col] = records[col].fillna(False).astype(bool)

    medication_records.append(records)


# 4. Combine medication records across survey cycles
all_meds = pd.concat(
    medication_records,
    ignore_index=True
)

print("\nTotal medication records:", len(all_meds))
print("Unique participants:", all_meds["SEQN"].nunique())


# 5. Group medication records by participant
grouped = all_meds.groupby("SEQN")

# Check for conflicting medication-use responses
status_counts = grouped["RX_USE"].nunique(dropna=True)

conflicting = status_counts[status_counts > 1]

if len(conflicting) > 0:
    print("\nConflicting medication-use responses:")
    print(conflicting.head(10))

    raise ValueError(
        "Review inconsistent RX_USE responses"
    )

status = grouped["RX_USE"].first()

# Identify participants with at least one unidentified drug
unknown_drug = grouped["RXDDRGID"].agg(
    lambda values: values.isna().any()
)

# Create one row per participant
med_features = pd.DataFrame({
    "SEQN": status.index,
    "RX_USE": status.values
}).set_index("SEQN")


# 6. Derive medication indicators
#
# 1 = Reported medication in this drug class
# 0 = No reported medication in this class,
#     with sufficient information to classify
# NA = Medication information incomplete or unknown

for col in flag_cols:

    positive = grouped[col].any()

    result = pd.Series(
        pd.NA,
        index=status.index,
        dtype="Int64"
    )

    # Confirmed no prescription medication use
    result.loc[status == 2] = 0

    # At least one reported medication in this class
    result.loc[
        (status == 1) & positive
    ] = 1

    # Medication use reported, all drug identities known,
    # but none belongs to this class
    result.loc[
        (status == 1)
        & ~positive
        & ~unknown_drug
    ] = 0

    med_features[col] = result


# 7. Restore SEQN as a column
med_features = med_features.reset_index()


# 8. Validate the resulting dataset
assert med_features["SEQN"].is_unique

assert set(med_features["SEQN"]).issubset(
    set(all_meds["SEQN"])
)

for col in flag_cols:
    assert med_features[col].dropna().isin([0, 1]).all()


# 9. Display results
print("\n===================================")
print("MEDICATION FEATURE SUMMARY")
print("===================================")

print("Medication records:", len(all_meds))
print("Unique participants:", len(med_features))

print("\nPrescription medication-use responses:")
print(
    med_features["RX_USE"].value_counts(
        dropna=False
    )
)

print("\nDerived medication predictors:")

for col in flag_cols:

    print(f"\n{col}")

    print(
        med_features[col].value_counts(
            dropna=False
        )
    )

print("\nFirst five participants:")
display(med_features.head())

print("\nMedication feature construction completed.")


Processing 1999-2000...
Total records: 15643
Records with drug IDs: 9138
Unmatched drug IDs: 0

Processing 2001-2002...
Total records: 18418
Records with drug IDs: 11447
Unmatched drug IDs: 0

Processing 2003-2004...
Total records: 18401
Records with drug IDs: 12231
Unmatched drug IDs: 0

Total medication records: 52462
Unique participants: 31126

MEDICATION FEATURE SUMMARY
Medication records: 52462
Unique participants: 31126

Prescription medication-use responses:
RX_USE
2.0    19280
1.0    11787
7.0       28
NaN       25
9.0        6
Name: count, dtype: int64

Derived medication predictors:

DERIVED_HL_MED
DERIVED_HL_MED
0       29060
1        1721
<NA>      345
Name: count, dtype: Int64

DERIVED_HTN_MED
DERIVED_HTN_MED
0       26653
1        4152
<NA>      321
Name: count, dtype: Int64

DERIVED_DM_MED
DERIVED_DM_MED
0       29493
1        1286
<NA>      347
Name: count, dtype: Int64

First five participants:


,SEQN,RX_USE,DERIVED_HL_MED,DERIVED_HTN_MED,DERIVED_DM_MED
0,1.0,1.0,0,0,0
1,2.0,1.0,0,0,0
2,3.0,2.0,0,0,0
3,4.0,1.0,0,0,0
4,5.0,1.0,0,1,0



Medication feature construction completed.


In [13]:
from pathlib import Path
import pandas as pd

# Load our DNA-enriched cohort
cohort_path = Path(
    "../results/cohort_features_with_dna_checkpoint.pkl"
)

cohort_base = pd.read_pickle(cohort_path)

med_cols = [
    "DERIVED_HL_MED",
    "DERIVED_HTN_MED",
    "DERIVED_DM_MED"
]

# Validate before merging
assert cohort_base["SEQN"].is_unique
assert med_features["SEQN"].is_unique
assert not set(med_cols).intersection(cohort_base.columns)

# Merge one medication record per participant
cohort_with_medications = cohort_base.merge(
    med_features[["SEQN", "RX_USE"] + med_cols],
    on="SEQN",
    how="left",
    validate="one_to_one",
    indicator=True
)

# Verify cohort integrity
assert len(cohort_with_medications) == len(cohort_base) == 9957
assert cohort_with_medications["SEQN"].is_unique
assert (cohort_with_medications["_merge"] == "both").all()

cohort_with_medications = cohort_with_medications.drop(
    columns="_merge"
)

# Verify no positive medication flag contradicts confirmed non-use
for col in med_cols:
    assert not (
        (cohort_with_medications["RX_USE"] == 2)
        & (cohort_with_medications[col] == 1)
    ).any()

print("Final participants:", len(cohort_with_medications))
print("Dataset columns:", cohort_with_medications.shape[1])

print("\nMedication indicators in our research cohort:")

for col in med_cols:
    print(f"\n{col}")
    print(
        cohort_with_medications[col].value_counts(
            dropna=False
        )
    )

# Save without overwriting earlier checkpoints
output = Path(
    "../results/cohort_features_with_dna_medications_checkpoint.pkl"
)

cohort_with_medications.to_pickle(output)

print("\nSaved:", output)

Final participants: 9957
Dataset columns: 299

Medication indicators in our research cohort:

DERIVED_HL_MED
DERIVED_HL_MED
0       8093
1       1693
<NA>     171
Name: count, dtype: Int64

DERIVED_HTN_MED
DERIVED_HTN_MED
0       5829
1       3977
<NA>     151
Name: count, dtype: Int64

DERIVED_DM_MED
DERIVED_DM_MED
0       8592
1       1192
<NA>     173
Name: count, dtype: Int64

Saved: ../results/cohort_features_with_dna_medications_checkpoint.pkl


In [15]:
from pathlib import Path
import pandas as pd

filename = "NHANES_Lin_2026_134_Variable_Map_Verified.xlsx"

locations = [
    Path("../") / filename,
    Path("../docs") / filename,
    Path.home() / "Downloads" / filename,
    Path.home() / "Documents" / filename
]

found = [p for p in locations if p.exists()]

if found:
    map_path = found[0]

    print("Found feature map:", map_path)

    workbook = pd.ExcelFile(map_path)

    print("\nAvailable sheets:")
    print(workbook.sheet_names)

    for sheet in workbook.sheet_names:
        df = pd.read_excel(map_path, sheet_name=sheet)

        print(f"\n--- {sheet} ---")
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())

        display(df.head(5))
else:
    print("Feature map not found in the checked folders.")

Found feature map: /Users/evodemanirahari/Downloads/NHANES_Lin_2026_134_Variable_Map_Verified.xlsx

Available sheets:
['All_134', 'OneHot_Only', 'Source_Files', 'README']

--- All_134 ---
Rows: 134
Columns: ['#', 'Supplement variable', 'NHANES code / derivation', 'Source file / component', 'Data class', 'One-hot?', 'Mapping status', 'Replication notes', 'Official CDC source']


,#,Supplement variable,NHANES code / derivation,Source file / component,Data class,One-hot?,Mapping status,Replication notes,Official CDC source
0,1,Gender,RIAGENDR,DEMO,Categorical,Yes,Verified direct,"1=Male, 2=Female; clean any special codes if p...",https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1...
1,2,Age,RIDAGEYR,DEMO,Numeric,No,Verified direct,Age in years; keep numeric.,https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1...
2,3,Race,RIDRETH1,DEMO,Categorical,Yes,Verified direct,Race/ethnicity recode; numeric values are cate...,https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1...
3,4,Income,INDHHINC,DEMO,Categorical,Yes,Verified direct,Annual household income is reported as ranges;...,https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1...
4,5,Resting heart rate,BPXPLS,BPX,Numeric,No,Verified direct,60-second pulse; closest direct NHANES measure...,https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1...



--- OneHot_Only ---
Rows: 14
Columns: ['#', 'Supplement variable', 'NHANES code / derivation', 'Source', 'Why categorical', 'Cleaning note']


,#,Supplement variable,NHANES code / derivation,Source,Why categorical,Cleaning note
0,1,Gender,RIAGENDR,DEMO,Nominal / binary category; paper says categori...,Convert refusal/don't-know/special nonresponse...
1,2,Race,RIDRETH1,DEMO,Nominal / binary category; paper says categori...,Convert refusal/don't-know/special nonresponse...
2,3,Income,INDHHINC,DEMO,Nominal / binary category; paper says categori...,Convert refusal/don't-know/special nonresponse...
3,4,Cardiovascular fitness level,CVDFITLV,CVX,Nominal / binary category; paper says categori...,Convert refusal/don't-know/special nonresponse...
4,5,Physical activity,PAQ180,PAQ,Nominal / binary category; paper says categori...,Convert refusal/don't-know/special nonresponse...



--- Source_Files ---
Rows: 22
Columns: ['Priority', 'Source/component', 'Example variables from Table S1', 'Official link', 'Notes']


,Priority,Source/component,Example variables from Table S1,Official link,Notes
0,Start now,DEMO,"Gender, Age, Race, Income",https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1...,You already have DEMO/DEMO_B/DEMO_C.
1,Start now,BPX,Resting heart rate,https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1...,Merge by SEQN.
2,Next,BMX,"BMI, lengths, circumferences, skinfolds",https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1...,Examination file.
3,Next,BIX,"ECF, TBW, ICF, fat-free mass, fat mass",https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1...,Age-limited examination -> substantial structu...
4,Next,CVX,"VO2, treadmill HR, fitness level",https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1...,Age 12-49 only -> heavy structural missingness...



--- README ---
Rows: 9
Columns: ['NHANES Lin et al. 2026 - 134-variable replication map', 'Unnamed: 1']


,NHANES Lin et al. 2026 - 134-variable replication map,Unnamed: 1
0,Purpose,Map the exact 134 features in Supplement Table...
1,Source of feature list,"User-provided supplement mmc1.pdf, Table S1. T..."
2,One-hot rule,Categorical variables are marked Yes. Fit impu...
3,Important status labels,Verified direct = CDC raw field matches the su...
4,Two total cholesterol rows,Table S1 lists Total cholesterol twice. This m...


In [16]:
import pandas as pd
from pathlib import Path

# Load the verified research variable map
feature_map = pd.read_excel(
    map_path,
    sheet_name="All_134"
)

# Load our latest research dataset
cohort_final = pd.read_pickle(
    "../results/cohort_features_with_dna_medications_checkpoint.pkl"
)

# Get the required variable names
feature_map["NHANES_Code"] = (
    feature_map["NHANES code / derivation"]
    .astype("string")
    .str.strip()
)

# Check exact column-name matches
available_columns = set(cohort_final.columns)

feature_map["Available"] = (
    feature_map["NHANES_Code"]
    .isin(available_columns)
)

feature_map["Status"] = feature_map["Available"].map({
    True: "Exact match found",
    False: "Needs mapping or derivation review"
})

print("NHANES RESEARCH FEATURE AUDIT")
print("-----------------------------")

print("Required variables:", len(feature_map))
print("Dataset columns:", cohort_final.shape[1])

print(
    "Exact matches:",
    feature_map["Available"].sum()
)

print(
    "Need review:",
    (~feature_map["Available"]).sum()
)

print("\nVariables needing review:")

display(
    feature_map.loc[
        ~feature_map["Available"],
        [
            "Supplement variable",
            "NHANES_Code",
            "Mapping status",
            "Status"
        ]
    ]
)

print("\nAudit summary:")
display(feature_map["Status"].value_counts())

NHANES RESEARCH FEATURE AUDIT
-----------------------------
Required variables: 134
Dataset columns: 299
Exact matches: 122
Need review: 12

Variables needing review:


,Supplement variable,NHANES_Code,Mapping status,Status
21,Heart rate increment,DERIVED_HR_INCREMENT,Derived - formula to confirm,Needs mapping or derivation review
22,Maximum heart rate,DERIVED_MAX_HR,Derived - formula to confirm,Needs mapping or derivation review
40,MET score,DERIVED_MET,Derived - formula to confirm,Needs mapping or derivation review
45,Androstanedione glucuronide,SS3adlG,Verified direct,Needs mapping or derivation review
90,Telomere length,TeloMean,Verified direct,Needs mapping or derivation review
91,SD of mean Telomere length,TeloStd,Verified direct,Needs mapping or derivation review
119,DNA methylation: NKcell,NKcell,Verified public DNAm,Needs mapping or derivation review
125,Glycated albumin,SSGALB,Strong match,Needs mapping or derivation review
126,eGFR,DERIVED_eGFR (from LBXSCR + demographics),Derived - formula to confirm,Needs mapping or derivation review
130,Taking hyperlipidemia medication,DERIVED_HL_MED (uses BPQ100D),Derived binary recode,Needs mapping or derivation review



Audit summary:


Status
Exact match found                     122
Needs mapping or derivation review     12
Name: count, dtype: int64

In [17]:
from pathlib import Path
import pandas as pd

# Review the 12 unmatched feature-map entries
review = feature_map.loc[
    ~feature_map["Available"],
    ["Supplement variable", "NHANES_Code"]
].copy()

# Remove explanatory notes from derived variable names
review["Short_Code"] = (
    review["NHANES_Code"]
    .str.split(" (", regex=False)
    .str[0]
)

existing = {
    col.lower(): col
    for col in cohort_final.columns
}

review["Matching_Column"] = review["Short_Code"].map(
    lambda code: existing.get(str(code).lower())
)

print("REVIEW OF 12 FEATURES")
display(review)

# Check the original questionnaire variables
questionnaire_cols = [
    "BPQ100D",
    "BPQ050A",
    "DIQ050",
    "DIQ070"
]

print("\nMedication questionnaire sources:")
for col in questionnaire_cols:
    print(col, ":", col in cohort_final.columns)

# Inspect downloaded files for missing laboratory features
raw_dir = Path("../data/raw")

sources = {
    "SSCARD_A.xpt": ["SSGALB"],
    "TELO_A.xpt": ["TeloMean", "TeloStd"],
    "TELO_B.xpt": ["TeloMean", "TeloStd"]
}

print("\nMissing features available in raw files:")

for filename, variables in sources.items():
    path = raw_dir / filename

    if not path.exists():
        print(filename, ": FILE NOT FOUND")
        continue

    data = pd.read_sas(path)

    found = [
        col for col in variables
        if col in data.columns
    ]

    print(filename)
    print("  Rows:", len(data))
    print("  Found:", found)

print("\nAudit complete.")

REVIEW OF 12 FEATURES


,Supplement variable,NHANES_Code,Short_Code,Matching_Column
21,Heart rate increment,DERIVED_HR_INCREMENT,DERIVED_HR_INCREMENT,NaN
22,Maximum heart rate,DERIVED_MAX_HR,DERIVED_MAX_HR,NaN
40,MET score,DERIVED_MET,DERIVED_MET,NaN
45,Androstanedione glucuronide,SS3adlG,SS3adlG,SS3ADLG
90,Telomere length,TeloMean,TeloMean,TELOMEAN
91,SD of mean Telomere length,TeloStd,TeloStd,TELOSTD
119,DNA methylation: NKcell,NKcell,NKcell,Nkcell
125,Glycated albumin,SSGALB,SSGALB,NaN
126,eGFR,DERIVED_eGFR (from LBXSCR + demographics),DERIVED_eGFR,NaN
130,Taking hyperlipidemia medication,DERIVED_HL_MED (uses BPQ100D),DERIVED_HL_MED,DERIVED_HL_MED



Medication questionnaire sources:
BPQ100D : True
BPQ050A : True
DIQ050 : False
DIQ070 : True

Missing features available in raw files:
SSCARD_A.xpt
  Rows: 21890
  Found: ['SSGALB']
TELO_A.xpt
  Rows: 3570
  Found: []
TELO_B.xpt
  Rows: 4269
  Found: []

Audit complete.


In [ ]:
import pandas as pd
from pathlib import Path

# Variables that need derivation or further inspection
search_terms = [
    "HR", "MET", "EGFR", "SCR",
    "CREAT", "VO2", "SSGALB"
]

print("POSSIBLE EXISTING VARIABLES")
print("---------------------------")

matches = [
    col for col in cohort_final.columns
    if any(term in col.upper() for term in search_terms)
]

review_existing = pd.DataFrame({
    "Variable": matches,
    "Available_Values": [
        int(cohort_final[col].notna().sum())
        for col in matches
    ],
    "Missing_Percent": [
        round(cohort_final[col].isna().mean() * 100, 1)
        for col in matches
    ]
})

display(review_existing)

# Inspect the source file for glycated albumin
cardiac = pd.read_sas(
    Path("../data/raw/SSCARD_A.xpt")
)

print("\nGLYCATED ALBUMIN SOURCE")
print("----------------------")

print("Records:", len(cardiac))
print("Unique participants:", cardiac["SEQN"].nunique())
print("Duplicate SEQN:", cardiac["SEQN"].duplicated().sum())

print("SSGALB measurements:",
      cardiac["SSGALB"].notna().sum())

print("Participants matching our cohort:",
      cardiac["SEQN"].isin(
          cohort_final["SEQN"]
      ).sum())

display(cardiac[["SEQN", "SSGALB"]].head())

POSSIBLE EXISTING VARIABLES
---------------------------


,Variable,Available_Values,Missing_Percent
0,DMDHRGND,9954,0.0
1,DMDHRAGE,9954,0.0
2,DMDHRBRN,9679,2.8
3,DMDHREDU,9678,2.8
4,DMDHRMAR,9502,4.6
5,MAX_HR,1023,89.7
6,SSGALBP,8001,19.6
7,serum_creatinine,8541,14.2
8,eGFR,8541,14.2
9,low_eGFR,8541,14.2



GLYCATED ALBUMIN SOURCE
----------------------
Records: 21890
Unique participants: 21890
Duplicate SEQN: 0
SSGALB measurements: 21842
Participants matching our cohort: 8022


,SEQN,SSGALB
0,2.0,0.545
1,3.0,0.497
2,5.0,0.544
3,6.0,0.637
4,7.0,0.630


In [19]:
import pandas as pd
from pathlib import Path

# Start from our latest DNA + medication checkpoint
cohort_base = pd.read_pickle(
    "../results/cohort_features_with_dna_medications_checkpoint.pkl"
)

# Glycated albumin source
cardiac = pd.read_sas("../data/raw/SSCARD_A.xpt")

# Check identifiers and prevent accidental duplicate columns
assert cohort_base["SEQN"].is_unique
assert cardiac["SEQN"].is_unique
assert "SSGALB" not in cohort_base.columns

# Keep only the measurement required by our feature map
glycated_albumin = cardiac[["SEQN", "SSGALB"]].copy()

# Merge without removing participants
cohort_updated = cohort_base.merge(
    glycated_albumin,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

# Verify the original cohort was preserved
assert len(cohort_updated) == 9957
assert cohort_updated["SEQN"].is_unique
assert cohort_updated["SSGALB"].notna().sum() > 0

print("Participants:", len(cohort_updated))
print("Dataset columns:", cohort_updated.shape[1])
print("Glycated albumin available:",
      cohort_updated["SSGALB"].notna().sum())

print("Glycated albumin missing:",
      cohort_updated["SSGALB"].isna().sum())

# Save as a new checkpoint
output = Path(
    "../results/cohort_features_with_glycated_albumin_checkpoint.pkl"
)

cohort_updated.to_pickle(output)

print("\nSaved:", output)

Participants: 9957
Dataset columns: 300
Glycated albumin available: 8012
Glycated albumin missing: 1945

Saved: ../results/cohort_features_with_glycated_albumin_checkpoint.pkl


In [20]:
# Inspect inputs for the remaining derived features

from pathlib import Path
import pandas as pd

columns_to_check = [
    "RIDAGEYR",
    "RIAGENDR",
    "serum_creatinine",
    "eGFR",
    "MAX_HR",
    "CVDWHR",
    "CVDS1HR",
    "CVDS2HR",
    "CVDR1HR",
    "CVDESVO2",
    "CVDVOMAX",
    "BPXPLS"
]

print("EXISTING COHORT VARIABLES")
print("-------------------------")

rows = []

for col in columns_to_check:
    if col in cohort_updated.columns:
        values = pd.to_numeric(
            cohort_updated[col], errors="coerce"
        )

        rows.append({
            "Variable": col,
            "Available": int(values.notna().sum()),
            "Missing_%": round(values.isna().mean() * 100, 1),
            "Minimum": values.min(),
            "Median": values.median(),
            "Maximum": values.max()
        })

display(pd.DataFrame(rows))

print("\nRAW CARDIOVASCULAR FITNESS FILES")
print("--------------------------------")

for filename in ["CVX.xpt", "CVX_B.xpt", "CVX_C.xpt"]:
    data = pd.read_sas(Path("../data/raw") / filename)

    relevant = [
        col for col in data.columns
        if "HR" in col or "VO2" in col
    ]

    print(f"\n{filename}")
    print("Relevant variables:", relevant)

EXISTING COHORT VARIABLES
-------------------------


,Variable,Available,Missing_%,Minimum,Median,Maximum
0,RIDAGEYR,9957,0.0,40.000000,62.000000,85.00000
1,RIAGENDR,9957,0.0,1.000000,2.000000,2.00000
2,serum_creatinine,8541,14.2,0.349600,0.900000,13.70000
3,eGFR,8541,14.2,2.653709,86.726289,130.59854
4,MAX_HR,1023,89.7,83.000000,129.000000,175.00000
5,CVDWHR,922,90.7,70.000000,102.000000,161.00000
6,CVDR1HR,950,90.5,77.000000,121.000000,168.00000
7,CVDESVO2,907,90.9,18.230000,37.610000,92.00000
8,CVDVOMAX,1140,88.6,1.200000,29.950000,50.17000
9,BPXPLS,8778,11.8,32.000000,70.000000,144.00000



RAW CARDIOVASCULAR FITNESS FILES
--------------------------------

CVX.xpt
Relevant variables: ['CVDPMHR', 'CVDWHR', 'CVDS1HR', 'CVDS2HR', 'CVDR1HR', 'CVDR2HR', 'CVDR3HR', 'CVDESVO2']

CVX_B.xpt
Relevant variables: ['CVDPMHR', 'CVDWHR', 'CVDS1HR', 'CVDS2HR', 'CVDR1HR', 'CVDR2HR', 'CVDR3HR', 'CVDESVO2']

CVX_C.xpt
Relevant variables: ['CVDPMHR', 'CVDWHR', 'CVDS1HR', 'CVDS2HR', 'CVDR1HR', 'CVDR2HR', 'CVDR3HR', 'CVDESVO2']


In [21]:
# Check how our existing MAX_HR compares with NHANES measurements

import numpy as np
import pandas as pd
from pathlib import Path

raw_dir = Path("../data/raw")

cvx_files = ["CVX.xpt", "CVX_B.xpt", "CVX_C.xpt"]

heart_rate_cols = [
    "SEQN",
    "CVDPMHR",
    "CVDWHR",
    "CVDS1HR",
    "CVDS2HR",
    "CVDR1HR"
]

cvx_data = []

for filename in cvx_files:
    data = pd.read_sas(raw_dir / filename)

    cvx_data.append(
        data[[c for c in heart_rate_cols if c in data.columns]]
    )

cvx_all = pd.concat(cvx_data, ignore_index=True)

assert cvx_all["SEQN"].is_unique

# Compare with our current cohort
hr_check = cohort_updated[
    ["SEQN", "RIDAGEYR", "MAX_HR"]
].merge(
    cvx_all,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

# Highest measured heart rate in the exercise stages
hr_check["STAGE_PEAK_HR"] = hr_check[
    ["CVDS1HR", "CVDS2HR"]
].max(axis=1)

# CDC age-predicted maximum heart rate
hr_check["AGE_PREDICTED_HR"] = (
    220 - hr_check["RIDAGEYR"]
)

def compare_hr(reference):
    paired = hr_check[
        ["MAX_HR", reference]
    ].dropna()

    if paired.empty:
        return 0, None

    exact = np.isclose(
        paired["MAX_HR"],
        paired[reference],
        atol=0.01
    )

    return len(paired), round(exact.mean() * 100, 1)

print("MAX_HR COMPARISON")
print("------------------")

for col in [
    "CVDPMHR",
    "AGE_PREDICTED_HR",
    "STAGE_PEAK_HR"
]:
    count, agreement = compare_hr(col)

    print(
        f"{col}: {count} comparable participants, "
        f"{agreement}% exact agreement"
    )

print("\nExample participants:")

display(
    hr_check.loc[
        hr_check["MAX_HR"].notna(),
        [
            "RIDAGEYR",
            "MAX_HR",
            "CVDPMHR",
            "CVDS1HR",
            "CVDS2HR",
            "STAGE_PEAK_HR",
            "AGE_PREDICTED_HR"
        ]
    ].head(10)
)

MAX_HR COMPARISON
------------------
CVDPMHR: 1023 comparable participants, 0.0% exact agreement
AGE_PREDICTED_HR: 1023 comparable participants, 0.0% exact agreement
STAGE_PEAK_HR: 1023 comparable participants, 100.0% exact agreement

Example participants:


,RIDAGEYR,MAX_HR,CVDPMHR,CVDS1HR,CVDS2HR,STAGE_PEAK_HR,AGE_PREDICTED_HR
1,49.0,126.0,171.0,107.0,126.0,126.0,171.0
24,43.0,107.0,177.0,107.0,NaN,107.0,177.0
39,43.0,115.0,177.0,96.0,115.0,115.0,177.0
41,40.0,101.0,180.0,101.0,NaN,101.0,180.0
64,40.0,156.0,180.0,125.0,156.0,156.0,180.0
107,48.0,102.0,172.0,102.0,NaN,102.0,172.0
108,42.0,128.0,178.0,110.0,128.0,128.0,178.0
114,42.0,130.0,178.0,110.0,130.0,130.0,178.0
129,45.0,121.0,175.0,101.0,121.0,121.0,175.0
134,49.0,143.0,171.0,143.0,NaN,143.0,171.0


In [22]:
# Review the paper mapping notes for remaining derived features

features_to_review = [
    "Heart rate increment",
    "Maximum heart rate",
    "MET score",
    "eGFR"
]

review_rows = feature_map[
    feature_map["Supplement variable"].isin(features_to_review)
]

for _, row in review_rows.iterrows():
    print("\n" + "=" * 50)
    print("Feature:", row["Supplement variable"])
    print("Code:", row["NHANES code / derivation"])
    print("Mapping status:", row["Mapping status"])
    print("Source:", row["Source file / component"])
    print("Notes:", row["Replication notes"])


Feature: Heart rate increment
Code: DERIVED_HR_INCREMENT
Mapping status: Derived - formula to confirm
Source: CVX
Notes: Likely derived from exercise heart-rate fields; exact study formula is not stated in Supplement Table S1.

Feature: Maximum heart rate
Code: DERIVED_MAX_HR
Mapping status: Derived - formula to confirm
Source: CVX
Notes: Likely derived from treadmill heart-rate measurements (e.g., stage 1/2); do not substitute CVDPMHR (220-age) without confirmation.

Feature: MET score
Code: DERIVED_MET
Mapping status: Derived - formula to confirm
Source: PAQ / PAQIAF
Notes: Paper says MET score was derived; exact formula is not in Table S1. CDC PAQ documentation provides suggested MET values but the study derivation should be reproduced from author methods/code if available.

Feature: eGFR
Code: DERIVED_eGFR (from LBXSCR + demographics)
Mapping status: Derived - formula to confirm
Source: LAB18 / biochemistry cycle files
Notes: No single raw NHANES eGFR field in these cycles; derive

In [2]:
import pandas as pd
from pathlib import Path

# 1. Load our latest saved NHANES cohort
cohort = pd.read_pickle(
    "../results/cohort_features_with_glycated_albumin_checkpoint.pkl"
)

print("Research cohort:", len(cohort))

# 2. Load the three physical activity datasets
raw_dir = Path("../data/raw")

files = [
    "PAQIAF.xpt",
    "PAQIAF_B.xpt",
    "PAQIAF_C.xpt"
]

activity_data = pd.concat(
    [pd.read_sas(raw_dir / filename) for filename in files],
    ignore_index=True
)

# 3. Keep only participants in our research cohort
activity_cohort = activity_data[
    activity_data["SEQN"].isin(cohort["SEQN"])
].copy()

# 4. Check activity data
print("\nACTIVITY DATA SUMMARY")
print("---------------------")

print("Activity records:", len(activity_cohort))
print(
    "Participants with activity records:",
    activity_cohort["SEQN"].nunique()
)

print("\nAvailable columns:")
print(activity_cohort.columns.tolist())

# 5. Check missing values
check_cols = ["PADMETS", "PADTIMES", "PADDURAT"]

print("\nMissing values:")
print(
    activity_cohort[
        [c for c in check_cols if c in activity_cohort.columns]
    ].isna().sum()
)

# 6. Show example activity records
sample_cols = [
    "SEQN",
    "PADACTIV",
    "PADLEVEL",
    "PADMETS",
    "PADTIMES",
    "PADDURAT"
]

print("\nExample activity records:")

display(
    activity_cohort[
        [c for c in sample_cols if c in activity_cohort.columns]
    ].head(10)
)

Research cohort: 9957

ACTIVITY DATA SUMMARY
---------------------
Activity records: 9536
Participants with activity records: 4741

Available columns:
['SEQN', 'PADACTIV', 'PADLEVEL', 'PADTIMES', 'PADDURAT', 'PADMETS', 'PAAQUEX']

Missing values:
PADMETS      0
PADTIMES    10
PADDURAT     9
dtype: int64

Example activity records:


,SEQN,PADACTIV,PADLEVEL,PADMETS,PADTIMES,PADDURAT
0,5.0,12.0,1.0,6.0,4.0,30.0
1,5.0,18.0,1.0,4.0,5.0,60.0
2,5.0,44.0,1.0,4.0,4.0,30.0
3,5.0,42.0,2.0,5.0,9.0,90.0
4,5.0,43.0,2.0,6.0,4.0,60.0
8,7.0,42.0,1.0,3.5,13.0,15.0
16,14.0,42.0,1.0,3.5,30.0,30.0
17,14.0,14.0,2.0,3.0,9.0,60.0
21,16.0,18.0,2.0,5.0,13.0,120.0
25,25.0,42.0,1.0,3.5,9.0,17.0


In [3]:
import pandas as pd
import numpy as np

# Work with the activity records already loaded
activity = activity_cohort.copy()

columns = ["PADMETS", "PADTIMES", "PADDURAT"]

for col in columns:
    activity[col] = pd.to_numeric(
        activity[col], errors="coerce"
    )

# Require valid intensity, frequency, and duration
valid = (
    activity["PADMETS"].between(2.5, 10)
    & activity["PADTIMES"].between(1, 300)
    & activity["PADDURAT"].between(10, 600)
)

activity["MET_MINUTES"] = np.where(
    valid,
    activity["PADMETS"]
    * activity["PADTIMES"]
    * activity["PADDURAT"],
    np.nan
)

# Aggregate activities into one row per participant
met_summary = activity.groupby("SEQN").agg(
    MET_MIN_30D=("MET_MINUTES", lambda x: x.sum(min_count=1)),
    Activity_Count=("SEQN", "size"),
    Incomplete_Activities=("MET_MINUTES", lambda x: x.isna().sum())
).reset_index()

# Do not report incomplete totals as complete measurements
met_summary.loc[
    met_summary["Incomplete_Activities"] > 0,
    "MET_MIN_30D"
] = np.nan

assert met_summary["SEQN"].is_unique

print("Participants with activity records:", len(met_summary))
print("Participants with complete MET totals:",
      met_summary["MET_MIN_30D"].notna().sum())
print("Participants with incomplete activity records:",
      (met_summary["Incomplete_Activities"] > 0).sum())

print("\nMET-minutes summary:")
display(met_summary["MET_MIN_30D"].describe().round(1))

print("\nExample participants:")
display(met_summary.head(10))

Participants with activity records: 4741
Participants with complete MET totals: 4732
Participants with incomplete activity records: 9

MET-minutes summary:


count      4732.0
mean       6342.6
std        9541.1
min          35.0
25%        1425.0
50%        3360.0
75%        7380.0
max      153140.0
Name: MET_MIN_30D, dtype: float64


Example participants:


,SEQN,MET_MIN_30D,Activity_Count,Incomplete_Activities
0,5.0,7890.0,5,0
1,7.0,682.5,1,0
2,14.0,4770.0,2,0
3,16.0,7800.0,1,0
4,25.0,535.5,1,0
5,55.0,1800.0,2,0
6,83.0,560.0,1,0
7,90.0,3480.0,4,0
8,93.0,840.0,1,0
9,95.0,1365.0,1,0


In [4]:
import pandas as pd
import numpy as np

# Work with the activity records already loaded
activity = activity_cohort.copy()

columns = ["PADMETS", "PADTIMES", "PADDURAT"]

for col in columns:
    activity[col] = pd.to_numeric(
        activity[col], errors="coerce"
    )

# Require valid intensity, frequency, and duration
valid = (
    activity["PADMETS"].between(2.5, 10)
    & activity["PADTIMES"].between(1, 300)
    & activity["PADDURAT"].between(10, 600)
)

activity["MET_MINUTES"] = np.where(
    valid,
    activity["PADMETS"]
    * activity["PADTIMES"]
    * activity["PADDURAT"],
    np.nan
)

# Aggregate activities into one row per participant
met_summary = activity.groupby("SEQN").agg(
    MET_MIN_30D=("MET_MINUTES", lambda x: x.sum(min_count=1)),
    Activity_Count=("SEQN", "size"),
    Incomplete_Activities=("MET_MINUTES", lambda x: x.isna().sum())
).reset_index()

# Do not report incomplete totals as complete measurements
met_summary.loc[
    met_summary["Incomplete_Activities"] > 0,
    "MET_MIN_30D"
] = np.nan

assert met_summary["SEQN"].is_unique

print("Participants with activity records:", len(met_summary))
print("Participants with complete MET totals:",
      met_summary["MET_MIN_30D"].notna().sum())
print("Participants with incomplete activity records:",
      (met_summary["Incomplete_Activities"] > 0).sum())

print("\nMET-minutes summary:")
display(met_summary["MET_MIN_30D"].describe().round(1))

print("\nExample participants:")
display(met_summary.head(10))

Participants with activity records: 4741
Participants with complete MET totals: 4732
Participants with incomplete activity records: 9

MET-minutes summary:


count      4732.0
mean       6342.6
std        9541.1
min          35.0
25%        1425.0
50%        3360.0
75%        7380.0
max      153140.0
Name: MET_MIN_30D, dtype: float64


Example participants:


,SEQN,MET_MIN_30D,Activity_Count,Incomplete_Activities
0,5.0,7890.0,5,0
1,7.0,682.5,1,0
2,14.0,4770.0,2,0
3,16.0,7800.0,1,0
4,25.0,535.5,1,0
5,55.0,1800.0,2,0
6,83.0,560.0,1,0
7,90.0,3480.0,4,0
8,93.0,840.0,1,0
9,95.0,1365.0,1,0


In [5]:
import pandas as pd
from pathlib import Path

raw_dir = Path("../data/raw")

# Load general physical activity questionnaires
paq_files = [
    "PAQ.xpt",
    "PAQ_B.xpt",
    "PAQ_C.xpt"
]

paq_data = pd.concat(
    [pd.read_sas(raw_dir / f) for f in paq_files],
    ignore_index=True
)

# Verify participant IDs
assert paq_data["SEQN"].is_unique

# Keep the relevant questionnaire responses
paq_review = cohort[["SEQN"]].merge(
    paq_data[["SEQN", "PAD200", "PAD320"]],
    on="SEQN",
    how="left",
    validate="one_to_one"
)

# Identify participants with activity records
paq_review["Has_Activity_Record"] = (
    paq_review["SEQN"].isin(met_summary["SEQN"])
)

# Focus on participants without activity records
no_records = paq_review[
    ~paq_review["Has_Activity_Record"]
].copy()

print("Participants without activity records:",
      len(no_records))

print("\nQuestionnaire responses:")
print("1 = Yes, 2 = No, 3 = Unable to do activity")
print("7 = Refused, 9 = Don't know")

display(
    pd.crosstab(
        no_records["PAD200"].fillna(-1),
        no_records["PAD320"].fillna(-1)
    )
)

# Count confirmed nonparticipants
confirmed_none = (
    no_records["PAD200"].eq(2)
    & no_records["PAD320"].eq(2)
)

# Check cases requiring further review
reported_yes = (
    no_records["PAD200"].eq(1)
    | no_records["PAD320"].eq(1)
)

print("\nConfirmed no moderate or vigorous activity:",
      confirmed_none.sum())

print("Reported activity but no activity record:",
      reported_yes.sum())

print("Other responses requiring review:",
      len(no_records)
      - confirmed_none.sum()
      - reported_yes.sum())

Participants without activity records: 5216

Questionnaire responses:
1 = Yes, 2 = No, 3 = Unable to do activity
7 = Refused, 9 = Don't know


PAD320,-1.0,2.0,3.0,7.0,9.0
PAD200,,,,,
-1.0,5,0,0,0,0
2.0,0,4418,36,0,2
3.0,0,114,630,0,0
7.0,0,0,0,9,0
9.0,0,0,0,0,2



Confirmed no moderate or vigorous activity: 4418
Reported activity but no activity record: 0
Other responses requiring review: 798


In [6]:
import pandas as pd

# Start with our original 9,957 participants
cohort_met = cohort[["SEQN"]].copy()

# Merge calculated MET totals
cohort_met = cohort_met.merge(
    met_summary[["SEQN", "MET_MIN_30D"]],
    on="SEQN",
    how="left",
    validate="one_to_one"
)

# Participants who explicitly reported no
# moderate or vigorous leisure-time activity
confirmed_zero_ids = no_records.loc[
    no_records["PAD200"].eq(2)
    & no_records["PAD320"].eq(2),
    "SEQN"
]

# Assign zero only to confirmed nonparticipants
cohort_met.loc[
    cohort_met["SEQN"].isin(confirmed_zero_ids),
    "MET_MIN_30D"
] = 0.0

# Verify the original cohort is unchanged
assert len(cohort_met) == 9957
assert cohort_met["SEQN"].is_unique

print("MET DATASET SUMMARY")
print("-------------------")

print("Total participants:", len(cohort_met))

print(
    "Participants with positive MET-minutes:",
    (cohort_met["MET_MIN_30D"] > 0).sum()
)

print(
    "Participants with zero MET-minutes:",
    (cohort_met["MET_MIN_30D"] == 0).sum()
)

print(
    "Participants with missing MET-minutes:",
    cohort_met["MET_MIN_30D"].isna().sum()
)

print("\nMET distribution:")
display(cohort_met["MET_MIN_30D"].describe())

print("\nHighest MET totals:")
display(
    cohort_met.nlargest(10, "MET_MIN_30D")
)

MET DATASET SUMMARY
-------------------
Total participants: 9957
Participants with positive MET-minutes: 4732
Participants with zero MET-minutes: 4418
Participants with missing MET-minutes: 807

MET distribution:


count      9150.000000
mean       3280.135355
std        7557.752150
min           0.000000
25%           0.000000
50%         210.000000
75%        3600.000000
max      153140.000000
Name: MET_MIN_30D, dtype: float64


Highest MET totals:


,SEQN,MET_MIN_30D
7509,23618.0,153140.0
2602,8108.0,144000.0
273,907.0,100800.0
8970,28147.0,95850.0
4724,14852.0,91260.0
4027,12733.0,90630.0
2388,7411.0,90000.0
8637,27143.0,87390.0
8999,28249.0,87360.0
3452,10833.0,87110.0


In [7]:
import pandas as pd
from pathlib import Path

# Paper's 134-variable mapping
map_path = (
    Path.home() / "Downloads"
    / "NHANES_Lin_2026_134_Variable_Map_Verified.xlsx"
)

paper = pd.read_excel(map_path, sheet_name="All_134")

# Our most recent saved dataset
cohort = pd.read_pickle(
    "../results/cohort_features_with_glycated_albumin_checkpoint.pkl"
)

assert len(paper) == 134
assert len(cohort) == 9957
assert cohort["SEQN"].is_unique

# Match column names without capitalization differences
available = {
    str(col).lower(): col
    for col in cohort.columns
}

# Known aliases that still need formula verification
aliases = {
    "DERIVED_MAX_HR": "MAX_HR",
    "DERIVED_EGFR": "eGFR"
}

rows = []

for _, record in paper.iterrows():

    code = str(
        record["NHANES code / derivation"]
    ).split(" (", 1)[0].strip()

    source = available.get(code.lower())

    if source is not None:
        match_type = "Column found"
    elif code.upper() in aliases:
        source = available.get(
            aliases[code.upper()].lower()
        )
        match_type = (
            "Derived alias - verify formula"
            if source is not None
            else "Not available"
        )
    else:
        match_type = "Not available"

    mapping_status = str(record["Mapping status"])

    rows.append({
        "Paper feature": record["Supplement variable"],
        "Paper code": code,
        "Dataset column": source,
        "Match": match_type,
        "Mapping status": mapping_status,
        "Needs verification": (
            source is None
            or "Derived" in mapping_status
            or "Strong match" in mapping_status
        )
    })

audit_134 = pd.DataFrame(rows)

print("PAPER FEATURE EXTRACTION AUDIT")
print("------------------------------")
print("Paper predictors:", len(audit_134))
print("Cohort participants:", len(cohort))
print(
    "Matched columns:",
    audit_134["Dataset column"].notna().sum()
)
print(
    "Unavailable columns:",
    audit_134["Dataset column"].isna().sum()
)

print("\nFeatures needing review:")
display(
    audit_134[audit_134["Needs verification"]]
)

# Save the audit, not a model-ready dataset
output = Path("../results/paper_134_feature_audit.csv")
audit_134.to_csv(output, index=False)

print("\nAudit saved:", output)

PAPER FEATURE EXTRACTION AUDIT
------------------------------
Paper predictors: 134
Cohort participants: 9957
Matched columns: 132
Unavailable columns: 2

Features needing review:


,Paper feature,Paper code,Dataset column,Match,Mapping status,Needs verification
20,Heart rate before exercise,CVDWHR,CVDWHR,Column found,Strong match,True
21,Heart rate increment,DERIVED_HR_INCREMENT,NaN,Not available,Derived - formula to confirm,True
22,Maximum heart rate,DERIVED_MAX_HR,MAX_HR,Derived alias - verify formula,Derived - formula to confirm,True
32,Triglyceride,LBXSTR,LBXSTR,Column found,Strong match,True
40,MET score,DERIVED_MET,NaN,Not available,Derived - formula to confirm,True
88,Iron,LBXSIR,LBXSIR,Column found,Strong match,True
125,Glycated albumin,SSGALB,SSGALB,Column found,Strong match,True
126,eGFR,DERIVED_eGFR,eGFR,Derived alias - verify formula,Derived - formula to confirm,True
129,Cerebrovascular event history,MCQ160F,MCQ160F,Column found,Strong match,True
130,Taking hyperlipidemia medication,DERIVED_HL_MED,DERIVED_HL_MED,Column found,Derived binary recode,True



Audit saved: ../results/paper_134_feature_audit.csv


In [8]:
import pandas as pd

cohort = pd.read_pickle(
    "../results/cohort_features_with_glycated_albumin_checkpoint.pkl"
)

hr = cohort[
    ["SEQN", "RIDAGEYR", "MAX_HR", "CVDWHR", "BPXPLS"]
].copy()

for col in ["MAX_HR", "CVDWHR", "BPXPLS"]:
    hr[col] = pd.to_numeric(hr[col], errors="coerce")

# Compare two possible definitions
hr["INCREMENT_FROM_WARMUP"] = (
    hr["MAX_HR"] - hr["CVDWHR"]
)

hr["INCREMENT_FROM_REST"] = (
    hr["MAX_HR"] - hr["BPXPLS"]
)

for col in ["INCREMENT_FROM_WARMUP", "INCREMENT_FROM_REST"]:
    print(f"\n{col}")
    print("Available:", hr[col].notna().sum())
    print("Negative values:", (hr[col] < 0).sum())
    display(hr[col].describe().round(1))

print("\nExample participants:")
display(
    hr.loc[
        hr["MAX_HR"].notna(),
        [
            "RIDAGEYR",
            "MAX_HR",
            "CVDWHR",
            "BPXPLS",
            "INCREMENT_FROM_WARMUP",
            "INCREMENT_FROM_REST"
        ]
    ].head(10)
)


INCREMENT_FROM_WARMUP
Available: 861
Negative values: 22


count    861.0
mean      27.2
std       14.4
min      -12.0
25%       18.0
50%       26.0
75%       37.0
max       85.0
Name: INCREMENT_FROM_WARMUP, dtype: float64


INCREMENT_FROM_REST
Available: 1023
Negative values: 0


count    1023.0
mean       59.9
std        16.1
min        16.0
25%        48.0
50%        59.0
75%        70.5
max       106.0
Name: INCREMENT_FROM_REST, dtype: float64


Example participants:


,RIDAGEYR,MAX_HR,CVDWHR,BPXPLS,INCREMENT_FROM_WARMUP,INCREMENT_FROM_REST
1,49.0,126.0,94.0,66.0,32.0,60.0
24,43.0,107.0,112.0,62.0,-5.0,45.0
39,43.0,115.0,82.0,56.0,33.0,59.0
41,40.0,101.0,98.0,64.0,3.0,37.0
64,40.0,156.0,104.0,86.0,52.0,70.0
107,48.0,102.0,101.0,66.0,1.0,36.0
108,42.0,128.0,93.0,64.0,35.0,64.0
114,42.0,130.0,100.0,72.0,30.0,58.0
129,45.0,121.0,96.0,70.0,25.0,51.0
134,49.0,143.0,120.0,66.0,23.0,77.0


In [9]:
 # Inspect participants whose exercise HR
# was lower than their warm-up HR

negative_hr = hr.loc[
    hr["INCREMENT_FROM_WARMUP"] < 0,
    [
        "SEQN",
        "RIDAGEYR",
        "CVDWHR",
        "MAX_HR",
        "BPXPLS",
        "INCREMENT_FROM_WARMUP"
    ]
].copy()

print("Negative warm-up increments:", len(negative_hr))

display(negative_hr)

print("\nExercise measurements by age:")
hr["AGE_GROUP"] = pd.cut(
    hr["RIDAGEYR"],
    bins=[39, 49, 59, 69, 120],
    labels=["40-49", "50-59", "60-69", "70+"]
)

display(
    hr.groupby("AGE_GROUP", observed=True).agg(
        Participants=("SEQN", "size"),
        Exercise_HR_Available=("MAX_HR", "count"),
        Warmup_HR_Available=("CVDWHR", "count")
    )
)

Negative warm-up increments: 22


,SEQN,RIDAGEYR,CVDWHR,MAX_HR,BPXPLS,INCREMENT_FROM_WARMUP
24,96.0,43.0,112.0,107.0,62.0,-5.0
388,1212.0,47.0,106.0,103.0,86.0,-3.0
480,1470.0,45.0,125.0,123.0,80.0,-2.0
740,2149.0,47.0,117.0,111.0,72.0,-6.0
1137,3387.0,42.0,114.0,111.0,64.0,-3.0
1235,3740.0,43.0,127.0,124.0,64.0,-3.0
1720,5314.0,41.0,114.0,110.0,88.0,-4.0
2879,8979.0,43.0,132.0,123.0,86.0,-9.0
3304,10358.0,46.0,128.0,127.0,100.0,-1.0
3395,10654.0,45.0,114.0,110.0,66.0,-4.0



Exercise measurements by age:


,Participants,Exercise_HR_Available,Warmup_HR_Available
AGE_GROUP,,,
40-49,2473,1023,922
50-59,1880,0,0
60-69,2329,0,0
70+,3275,0,0
